[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.en/cap06/cap06_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

# 6 Document Analysis and Industrial Inspection

In **Part I — Digital Image Processing (DIP)**, techniques for image transformation and enhancement were studied, such as morphological operations, spatial filtering, convolutions, thresholding, segmentation, and frequency-domain processing.

**Part II — Computer Vision (CV)** expands this scope by addressing the automatic interpretation of visual content, involving information extraction, pattern recognition, and decision-making from images.

This chapter presents this transition through two representative applications:

1. **Automated Document Analysis**, applied to the processing of forms, assessments, and other structured documents through optical mark recognition (OMR) systems;
2. **Automated Industrial Inspection**, focused on quality control and defect detection in production lines.

These applications integrate techniques for geometric structure detection, extraction of invariant descriptors, pattern recognition, and object classification, forming the foundation of various modern visual inspection and automation systems.

## 6.1 Chapter Objectives

By the end of this chapter, the student should be able to:

* **Evaluate the influence of preprocessing** on the quality of automatic information recognition in documents;
* **Perform optical character recognition** (OCR) to convert scanned documents into encoded text;
* **Apply natural language processing techniques**, including machine translation, to the text obtained through OCR;
* Apply techniques for **geometric document alignment and rectification** using the Hough Transform and projective transformations;
* **Implement optical mark recognition** (OMR) systems for the automated reading of assessments and forms;
* **Detect and segment regions of interest** based on morphological operations, contours, and geometric properties;
* **Decode two-dimensional markers and barcodes**, integrating Computer Vision libraries into document processing *pipelines*;
* **Develop Computer Vision *pipelines*** for automated document analysis.

This chapter marks the transition from **Digital Image Processing**, focused on image transformation, to **Computer Vision**, whose goal is to interpret visual content, extract information, and support automated analysis and decision-making processes.

## 6.2 Environment Setup

The examples in this chapter use libraries widely employed in
CV-IP. The block below
installs the necessary packages; in environments that already have them, the execution
can be skipped.

In [1]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup()
from morph import mm

# install more dependencies beyond morph.py for this chapter
import sys, subprocess, importlib, shutil

def setup_cap06():
    """Installs system and Python-specific dependencies for Chapter 6
    (OCR, PDF reading, barcodes)."""

    # 1. System dependencies
    if 'google.colab' in sys.modules:
        print("[ENVIRONMENT] Google Colab. Setting up system dependencies...")
        subprocess.run(
            "apt-get update && apt-get install -y poppler-utils "
            "libzbar0 tesseract-ocr tesseract-ocr-por",
            shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
    elif shutil.which("tesseract") is None:
        # Local environment without tesseract: tries to install via apt-get (requires sudo/root)
        if shutil.which("apt-get"):
            print("[ENVIRONMENT] Local. Installing tesseract-ocr via apt-get (may ask for password)...")
            resultado = subprocess.run(
                "sudo apt-get update && sudo apt-get install -y tesseract-ocr tesseract-ocr-por",
                shell=True
            )
            if resultado.returncode != 0 or shutil.which("tesseract") is None:
                print(
                    "[WARNING] Could not install automatically. "
                    "Install manually: sudo apt install tesseract-ocr tesseract-ocr-por"
                )
        else:
            print(
                "[WARNING] tesseract not found and apt-get unavailable. "
                "Install manually before running the OCR cells."
            )

    # 2. Python dependencies (installs only missing ones)
    pkgs = {
        "cv2": "opencv-python", "skimage": "scikit-image", "numpy": "numpy",
        "pdf2image": "pdf2image", "pandas": "pandas", "tabulate": "tabulate",
        "PyPDF2": "PyPDF2", "bcrypt": "bcrypt", "pyarrow": "pyarrow",
        "pyzbar": "pyzbar", "pytesseract": "pytesseract", "deep_translator": "deep-translator"
    }
    for mod, pkg in pkgs.items():
        if importlib.util.find_spec(mod) is None:
            resultado_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
            if resultado_pip.returncode != 0:
                print(f"[WARNING] Failed to install {pkg} (required for module {mod}).")


setup_cap06()

# 3. Global pipeline imports
import cv2, numpy as np, matplotlib.pyplot as plt
from skimage import io, data, color

✅ Environment ready. Morph: 1.1.9 | OpenCV: 5.0.0
[ENVIRONMENT] Local. Installing tesseract-ocr via apt-get (may ask for password)...
[WARNING] Could not install automatically. Install manually: sudo apt install tesseract-ocr tesseract-ocr-por


sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: uma senha é necessária


In addition to these libraries, the didactic module `morph.py` will be used,
developed to simplify image reading, visualization, and processing operations
throughout this book. The following code checks its availability, performs the
*download* when necessary, and confirms the loaded version.

In [2]:
import os
import urllib.request

if not os.path.exists("morph.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/morph.py",
        "morph.py",
    )

import morph
from morph import mm

print(f"✅ Environment ready. morph {getattr(morph, '__version__', 'local_file')}")

✅ Environment ready. morph 1.1.9


## 6.3 Image Databases for Experimentation

The examples presented in this part of the book use, whenever possible, scanned documents, answer sheets, barcodes, *QRCodes*, and other images from real-world applications. To make the experiments fully reproducible — including in environments without internet access or access to the original MCTest files — we also employ public images widely used in teaching and research in Digital Image Processing and Computer Vision (DIP-CV).

> ### 💡 Why Use a Benchmark Image Database?
>
> Images such as `camera()` and `coins()` have been used for decades in textbooks, scientific articles, and teaching materials in DIP-CV. Their use offers important advantages:
>
> - **Reproducibility:** any reader obtains exactly the same images, regardless of the computer or operating system used, without requiring external downloads or files specific to this book;
> - **Comparability:** the results produced can be compared directly with those reported in the literature, since the same image sets are widely adopted as a reference;
> - **Focus on algorithms:** because these images are compact, well-documented, and distributed without usage restrictions for educational and scientific purposes, they allow attention to be concentrated on processing techniques, reducing interference related to data acquisition or management.

### 6.3.1 Public Images with `skimage.data`

The `skimage.data` module provides a collection of reference images widely used in teaching, research, and algorithm validation in DIP-CV. Inspecting the `data.__all__` attribute shows that the current version includes **42 items**, encompassing natural photographs, scanned documents, medical images, microscopy, textures, synthetic patterns, three-dimensional models, and time sequences. It is worth noting that some of these items correspond to utility functions, such as `data_dir()` and `download_all()`, rather than images proper.

Since the goal of this chapter is to present applications of document analysis and visual inspection, [Table 6.2](#tbl-06-skimage-data) compiles a **representative selection** of the most relevant images, organized according to their main applications in Computer Vision. [Figure 6.1](#fig-06-skimage-data) presents a sample of these images, grouped according to the same classification adopted in the table.

> ### 📝 Images used in this chapter
>
> Although `skimage.data` provides dozens of reference images, only four are directly employed in the experiments of this chapter. They were selected because they reproduce, in a controlled manner, characteristics frequently found in scanned documents and industrial visual inspection systems. [Table 6.1](#tbl-06-skimage-cap06) summarizes the role of each of them throughout this chapter.
>
> | Image | Application in the chapter |
> |---|---|
> | `data.page()` | Scanned page used in experiments on illumination correction, local enhancement (CLAHE), and automatic Otsu thresholding. |
> | `data.text()` | Document containing printed text, employed to illustrate segmentation, contour extraction, and typical OCR and OMR steps. |
> | `data.coffee()` | Color photograph with natural illumination variations, used to exemplify techniques applicable to non-documental real scenes. |
> | `data.brick()` | Reference texture employed in examples of surface inspection and defect detection through local variance analysis. |
>
> : Images from `skimage.data` used in the experiments of this chapter. {#tbl-06-skimage-cap06}

In [3]:
# @title { display-mode: "form" }
import pandas as pd
from IPython.display import Markdown

# Each category is associated with a color, reused in the titles of the images in @fig-06-skimage-data
categorias = {
    "📄 Documentos & OCR/OMR": {
        "cor": "#2563eb",
        "itens": [
            ("`data.page()`", "Página digitalizada de documento — normalização de fundo, CLAHE e Otsu."),
            ("`data.text()`", "Texto impresso — segmentação e extração de contornos em cenários de OCR/OMR."),
        ],
    },
    "🔵 Segmentação, Morfologia & Contornos": {
        "cor": "#16a34a",
        "itens": [
            ("`data.coins()`", "Conjunto de moedas — referência clássica para segmentação e *watershed*."),
            ("`data.clock()`", "Relógio analógico — detecção de formas e contornos."),
            ("`data.binary_blobs()`", "Blobs binários sintéticos — conectividade e morfologia matemática."),
            ("`data.moon()`", "Superfície lunar — segmentação de crateras por relevo de intensidade."),
        ],
    },
    "🧵 Textura & Inspeção Industrial": {
        "cor": "#ea580c",
        "itens": [
            ("`data.brick()`", "Textura uniforme de tijolos — detecção de defeitos por variância local."),
            ("`data.checkerboard()`", "Padrão xadrez — calibração de câmera e transformações geométricas."),
        ],
    },
    "🖼️ Fotografias Clássicas de PDI/VC": {
        "cor": "#7c3aed",
        "itens": [
            ("`data.camera()`", "Fotógrafo com tripé — imagem de referência mais citada na literatura de PDI."),
            ("`data.astronaut()`", "Retrato colorido de astronauta — filtragem e realce em cor."),
            ("`data.coffee()`", "Xícara de café — cena real com variação de iluminação e cor."),
            ("`data.cat()` / `data.chelsea()`", "Fotografias coloridas de gatos — detecção de bordas e realce."),
            ("`data.horse()`", "Silhueta binária de cavalo — descritores de forma e contorno."),
        ],
    },
}

linhas = []
for cat, info in categorias.items():
    for funcao, desc in info["itens"]:
        linhas.append({"Categoria": cat, "Função": funcao, "Descrição": desc})

df = pd.DataFrame(linhas)
Markdown(df.to_markdown(index=False, colalign=("left", "left", "left")))


**Table 6.2:** Selection of representative public images available in the *skimage.data* module, organized by application area.


| Categoria                              | Função                          | Descrição                                                                    |
|:---------------------------------------|:--------------------------------|:-----------------------------------------------------------------------------|
| 📄 Documentos & OCR/OMR                | `data.page()`                   | Página digitalizada de documento — normalização de fundo, CLAHE e Otsu.      |
| 📄 Documentos & OCR/OMR                | `data.text()`                   | Texto impresso — segmentação e extração de contornos em cenários de OCR/OMR. |
| 🔵 Segmentação, Morfologia & Contornos | `data.coins()`                  | Conjunto de moedas — referência clássica para segmentação e *watershed*.     |
| 🔵 Segmentação, Morfologia & Contornos | `data.clock()`                  | Relógio analógico — detecção de formas e contornos.                          |
| 🔵 Segmentação, Morfologia & Contornos | `data.binary_blobs()`           | Blobs binários sintéticos — conectividade e morfologia matemática.           |
| 🔵 Segmentação, Morfologia & Contornos | `data.moon()`                   | Superfície lunar — segmentação de crateras por relevo de intensidade.        |
| 🧵 Textura & Inspeção Industrial       | `data.brick()`                  | Textura uniforme de tijolos — detecção de defeitos por variância local.      |
| 🧵 Textura & Inspeção Industrial       | `data.checkerboard()`           | Padrão xadrez — calibração de câmera e transformações geométricas.           |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.camera()`                 | Fotógrafo com tripé — imagem de referência mais citada na literatura de PDI. |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.astronaut()`              | Retrato colorido de astronauta — filtragem e realce em cor.                  |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.coffee()`                 | Xícara de café — cena real com variação de iluminação e cor.                 |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.cat()` / `data.chelsea()` | Fotografias coloridas de gatos — detecção de bordas e realce.                |
| 🖼️ Fotografias Clássicas de PDI/VC     | `data.horse()`                  | Silhueta binária de cavalo — descritores de forma e contorno.                |

In [4]:
import matplotlib.pyplot as plt
from skimage import data

# Images ordered by category; the title color reproduces the category color in the previous tab.
imgs = [
    ("page",         data.page(),         "#2563eb"),   # Documents & OCR/OMR
    ("text",         data.text(),         "#2563eb"),
    ("coins",        data.coins(),        "#16a34a"),   # Segmentation & morphology
    ("binary_blobs", data.binary_blobs(), "#16a34a"),
    ("brick",        data.brick(),        "#ea580c"),   # Texture & industrial inspection
    ("checkerboard", data.checkerboard(),"#ea580c"),
    ("camera",       data.camera(),       "#7c3aed"),   # Classic computer vision/image processing photographs
    ("coffee",       data.coffee(),       "#7c3aed"),
]

fig, ax = plt.subplots(2, 4, figsize=(11, 5.5))
for a, (nome, img, cor) in zip(ax.ravel(), imgs):
    a.imshow(img, cmap="gray")
    a.set_title(nome, color=cor, fontweight="bold")
    a.axis("off")
plt.tight_layout()


<Figure size 3300x1650 with 8 Axes>

**Figure 6.1:** Sample of public images from *skimage.data*, grouped by application area.


## 6.4 Background Normalization and Local Contrast Equalization

Segmentation quality depends directly on the characteristics of the input image. In scanned documents, illumination variations, shadows, overexposed regions, and differences in paper tone reduce the contrast between the foreground and the background, making it difficult to apply global thresholding methods, such as the Otsu algorithm.

To minimize these effects, two complementary preprocessing techniques are employed:

- **Background normalization:** estimates the low-frequency component of the image through strong smoothing and then normalizes the original image with respect to this estimated background. This procedure reduces illumination gradients and compensates for slow intensity variations while preserving the structures of interest.
- **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*), presented in **Chapter 4:** divides the image into small regions (*tiles*) and performs histogram equalization for each region independently. Contrast is limited to avoid excessive noise amplification, making the technique especially suitable for images with local illumination variations.

[Figure 6.2](#fig-06-clahe-page) compares these strategies using the `page()` image from the `skimage.data` library. Six results are presented: (a) the original image; (b) direct binarization using the Otsu method, used as a reference; (c) the background estimated by Gaussian filtering; (d) the image after background normalization; (e) the binarization obtained after applying CLAHE followed by the Otsu method; and (f) the binarization obtained after background normalization followed by the application of the Otsu method.

The comparison allows one to observe the effect produced by each preprocessing stage and its influence on segmentation quality. In particular, background normalization reduces global illumination variations, while CLAHE increases local contrast between characters and background. Depending on the image characteristics, one or the other strategy may yield superior results, and there is no universally more suitable technique.

In [5]:
# |fig-cap: "Comparison between preprocessing strategies for text image binarization: (a) original image; (b) direct thresholding by Otsu's method; (c) background estimated by Gaussian filtering; (d) image after background normalization (division by the smoothed image); (e) CLAHE followed by Otsu thresholding; and (f) background normalization followed by Otsu thresholding. The images illustrate the effect of each technique on compensating for illumination variations and on segmentation quality."

import cv2
from skimage import data
from morph import mm

img = data.page()  # or: img = mm.gray(img_final) — with exam sheet image

# ── Method 1: Background normalization + Otsu ────────────────────────────────
# Estimates the background with a large-sigma Gaussian filter (slow light variations)
# and divides pixel by pixel to cancel the illumination gradient
bg = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
img_norm = cv2.divide(img, bg, scale=255)
img_norm_otsu = mm.threshold(img_norm)

# ── Method 2: CLAHE + Otsu ────────────────────────────────────────────────
# tileGridSize defines the size of each local region (tile)
# clipLimit controls the amplification ceiling — high values increase contrast
# but also amplify noise
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
img_clahe_otsu = mm.threshold(img_clahe)

# ── Method 0: Direct Otsu (no preprocessing) — reference ───────────
img_otsu = mm.threshold(img)

mm.show(
    [img,  img_otsu, bg, img_norm, img_clahe_otsu, img_norm_otsu],
    titles=["(a) Original", "(b) Direct Otsu", "(c) Gaussian",  \
            "(d) Normalized background (img/bg)", "(e) CLAHE + Otsu", \
            "(f) Normalized background + Otsu"],
    cols=3,
    figsize=(14, 8)
)


<Figure size 2100x1200 with 6 Axes>

**Figure 6.2**


> ### 📝 🧠 Why It Works — Background Normalization vs. CLAHE
>
> **Background normalization:** by dividing the image by a heavily smoothed version
> of itself, slow variations in illumination (light gradient, edge shadow) are
> eliminated without affecting fine details — text, lines, bubbles.
> The result is an image with approximately uniform lighting, where Otsu's global
> threshold works well across the entire page.
>
> **CLAHE:** a globally equalized histogram "stretches" the tones of the entire
> image at once — useful when lighting is uniform, but problematic when it is not.
> CLAHE divides the image into small blocks (*tiles*) and equalizes each one
> separately, with a maximum amplification limit (`clipLimit`) to avoid amplifying
> noise. It is especially effective for enhancing locally underexposed regions,
> but it does not eliminate global gradients — thus, applying it after background
> normalization tends to produce more consistent results.

## 6.5 Optical Character Recognition (OCR)

After binarization, the next stage in document processing consists of converting the visual representation of characters into digitally encoded text, a process known as **Optical Character Recognition** (**OCR**).

In general terms, an OCR system comprises three stages:

1. **Segmentation:** identifies lines, words, and characters in the image, using horizontal and vertical projections or connected-component detection.
2. **Feature extraction:** represents each character through visual attributes, such as edges, curvatures, and stroke patterns.
3. **Recognition:** associates the extracted attributes with the most likely character. Current systems predominantly use recurrent neural networks (LSTM) or architectures based on *transformers*.

In this book, we use **Tesseract OCR**, accessed through the `pytesseract` library (installation: `pip install pytesseract`). Originally developed by Hewlett-Packard between 1985 and 1995 and currently maintained by Google, Tesseract is described in Smith (2007) and Smith (2013). In recent versions, text recognition is performed by LSTM neural networks.

OCR performance depends on the quality of the input image. Noise, low contrast, geometric distortions, and uneven illumination reduce the recognition rate. For this reason, steps such as deskewing, background normalization, and adaptive equalization (CLAHE) are part of image preprocessing.

> ### 📝 🧠 Does Tesseract require a binarized image?
>
> Tesseract internally incorporates an adaptive binarization step before character recognition. For this reason, providing the OCR with a previously binarized image does not always yield the best results.
>
> Since thresholding is an irreversible operation, it may eliminate subtle intensity variations at character edges, such as *antialiasing*, which can assist the recognition mechanism. In many cases, a grayscale image with good illumination and contrast produces a more faithful transcription than its binarized version.

To investigate this effect, we compare the text extracted by Tesseract from four versions of the same image, presented in [Figure 6.3](#fig-06-ocr-comparacao): (a) original image; (b) image subjected to adaptive equalization (CLAHE) followed by thresholding using Otsu's method; (c) image subjected to background normalization followed by thresholding using Otsu's method; and (d) image subjected only to background normalization, preserving grayscale tones.

The comparison between versions (c) and (d) shows that, in passages containing visually similar characters, the grayscale version (d) produced a transcription more faithful to the original text than the binarized version (c). This result indicates that thresholding applied during preprocessing can eliminate information useful for recognition. Thus, although binarization is essential for several image processing operations, it does not necessarily constitute the best input for OCR. The choice of preprocessing technique should consider the subsequent stage of the document pipeline.

In [6]:
import pytesseract
import shutil as _sh
if _sh.which("tesseract") is None:
    # Build environment without Tesseract installed (e.g., without apt/sudo):
    # degrades instead of breaking the render. On Colab/local with Tesseract,
    # nothing changes.
    _AVISO_OCR = "[Tesseract OCR indisponivel neste ambiente - texto omitido]"
    pytesseract.image_to_string = lambda *a, **k: _AVISO_OCR
from skimage import data
import cv2
from morph import mm

img = data.page()

# ── Reusing the results from the previous section ────────────────────────
img_otsu = mm.threshold(img)

bg = cv2.GaussianBlur(img, (0, 0), sigmaX=25)
img_norm = cv2.divide(img, bg, scale=255)       # grayscale, without Otsu
img_norm_otsu = mm.threshold(img_norm)          # binarized

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
img_clahe_otsu = mm.threshold(img_clahe)

# ── Tesseract configuration ──────────────────────────────────────────────
# --psm 6: assumes a single uniform block of text (suitable for the `page` image)
config = "--psm 6"

texto_original   = pytesseract.image_to_string(img, config=config)
texto_clahe_otsu = pytesseract.image_to_string(img_clahe_otsu, config=config)
texto_norm_otsu  = pytesseract.image_to_string(img_norm_otsu, config=config)
texto_norm_gray  = pytesseract.image_to_string(img_norm, config=config)

for nome, texto in zip(
    ["(a) Original", "(b) CLAHE + Otsu", "(c) Normaliz. fundo + Otsu", 
     "(d) Normaliz. fundo (tons de cinza)"],
    [texto_original, texto_clahe_otsu, texto_norm_otsu, texto_norm_gray]
):
    print(f"--- {nome} ---")
    print(texto.strip(), "\n")

mm.show(
    [img, img_clahe_otsu, img_norm_otsu, img_norm],
    titles=["(a) Original", "(b) CLAHE + Otsu", "(c) Background norm. + Otsu", 
            "(d) Background norm. (grayscale)"],
    cols=4,
    figsize=(16, 4)
)

--- (a) Original ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (b) CLAHE + Otsu ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (c) Normaliz. fundo + Otsu ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 

--- (d) Normaliz. fundo (tons de cinza) ---
[Tesseract OCR indisponivel neste ambiente - texto omitido] 



<Figure size 2400x600 with 4 Axes>

**Figure 6.3:** Comparison of the text extracted by Tesseract from four versions of the same image: (a) original image; (b) CLAHE followed by Otsu thresholding; (c) background normalization followed by Otsu thresholding; and (d) background normalization in grayscale, without thresholding. The comparison shows that external binarization does not always favor recognition, since Tesseract already performs its own adaptive binarization internally.


> ### 📝 🧠 Why does preprocessing improve OCR?
>
> OCR performance directly depends on the quality of the input image. Low contrast, uneven illumination, noise, and geometric distortions hinder the separation between text and background and increase the likelihood of recognition errors.
>
> Techniques such as background normalization and adaptive equalization (CLAHE) correct illumination gradients and enhance local contrast between characters and background, producing images that are more suitable for automatic recognition. Thresholding, in turn, should be applied with caution: as it is an irreversible operation, it can eliminate subtle intensity variations at character edges—such as *antialiasing*—which the OCR engine itself uses internally to resolve ambiguities between visually similar symbols. For this reason, grayscale images, corrected only for illumination, often produce more faithful transcriptions than their binarized versions.
>
> In documents captured by mobile device cameras, preprocessing tends to provide greater performance gains than in documents scanned with a *scanner*, where illumination is usually more uniform.

## 6.6 Automatic Translation of Recognized Text

After optical character recognition, the resulting text can be subjected to natural language processing techniques, such as spell checking, indexing, summarization, and machine translation.

Machine translation constitutes an independent stage of OCR. While OCR converts the characters present in the image into encoded text, translation operates on that text in the document's original language. Thus, recognition errors can be propagated to the translation, compromising the quality of the result. Current machine translation systems predominantly employ neural architectures based on attention mechanisms and *transformers* [@Bahdanau2015; Vaswani (2017)].

In this example, the text obtained from the image subjected only to background normalization, without thresholding (item **d** of [Figure 6.3](#fig-06-ocr-comparacao)), is used, as it presents the most faithful transcription among the strategies evaluated in the previous section.

The translation is carried out using the `deep-translator` library (installation: `pip install deep-translator`), which provides an interface to different machine translation services, including Google Translate.

In [7]:
from deep_translator import GoogleTranslator
import shutil as _sh

# Text obtained by OCR from the image with background normalization (grayscale)
texto_en = texto_norm_gray

if _sh.which("tesseract") is None:
    # Build environment without Tesseract installed (see previous cell): text_en
    # is already the OCR unavailable placeholder, not real text — skip translation
    # instead of failing when trying to translate a string that is not really English.
    texto_pt = "[Tradução indisponível neste ambiente - Tesseract OCR ausente]"
else:
    texto_pt = GoogleTranslator(source="en", target="pt").translate(texto_en)

print("--- Original text generated by the normalized grayscale image (OCR, EN) ---")
print(texto_en)

print("\n--- Translated text (PT-BR) ---")
print(texto_pt)

mm.show(
    [img_norm],
    titles=["Image with background normalization (grayscale)"],
    cols=1,
    figsize=(6, 4)
)

--- Original text generated by the normalized grayscale image (OCR, EN) ---
[Tesseract OCR indisponivel neste ambiente - texto omitido]

--- Translated text (PT-BR) ---
[Tradução indisponível neste ambiente - Tesseract OCR ausente]


<Figure size 900x600 with 1 Axes>

**Figure 6.4:** Automatic recognition and translation flow. The image pre-processed by background normalization, without thresholding, is used as input to Tesseract OCR, and the recognized text is translated from English to Portuguese using the *deep-translator* library.


> ### 📝 🧠 Why does OCR quality influence translation?
>
> Machine translation uses as input the text produced by OCR. Recognition errors, such as incorrect characters, incomplete, or fragmented words, are propagated to the translation stage and can alter the meaning of the text.
>
> Consequently, the quality of the translation depends directly on the fidelity of the transcription obtained by OCR. As discussed previously, this fidelity is not always maximized by external binarization: grayscale images, corrected only for illumination, can preserve relevant information for distinguishing visually similar characters. Thus, image preprocessing—and the appropriate choice of its steps based on the subsequent task—contributes to improving not only character recognition but also the performance of later natural language processing stages, such as translation, indexing, and summarization.

## 6.7 Fundamentals of OMR and Industrial Inspection

**Optical Mark Recognition** (OMR) is a Computer Vision technique aimed at the automatic identification of markings in predefined positions on a form. Its applications include answer sheets, questionnaires, administrative forms, and other structured documents.

Unlike OCR (*Optical Character Recognition*), which recognizes characters and words, OMR determines the presence, absence, or intensity of marks in previously known regions. Instead of interpreting text, it exploits geometric and statistical properties associated with the filling of these regions.

Modern OMR systems process images obtained from *scanners*, cameras, or mobile devices, automating tasks that previously relied on specialized equipment.

In general terms, an OMR system comprises the following stages:

1. **Acquisition:** conversion of the physical document into digital format;
2. **Pre-processing:** geometric correction, noise reduction, and binarization;
3. **Location of regions of interest:** identification of the areas intended for markings;
4. **Analysis of markings:** assessment of the filling of candidate regions;
5. **Interpretation:** conversion of markings into responses or structured data.

These principles extend naturally to **Automated Industrial Inspection**.
On production lines, the same sequence — acquisition, pre-processing,
segmentation, feature extraction, and decision — is employed to
detect surface defects, verify component integrity, and
measure dimensions with subpixel precision. The difference lies in the application
domain: while OMR operates on documents with a predefined structure,
industrial inspection deals with objects whose geometric and
radiometric variations must be modeled more flexibly.

In the following sections, both applications are developed through
practical projects that reproduce typical stages of real systems.

## 6.8 Practical Projects: Building a Document Analysis *Pipeline*

The concepts in this chapter will be developed through projects that reproduce typical stages of real document analysis systems, introducing techniques reusable in OCR, OMR, visual inspection, and form processing applications.

### 6.8.1 Automatic Document Alignment (*OCR/OMR Pre-processing*)

Deskew correction is a fundamental stage in document processing. Rotations introduced during scanning or capture compromise the location of regions of interest and reduce the accuracy of subsequent stages.

In this project, a system will be developed to automatically estimate the predominant orientation of the document and correct its skew. To this end, classical edge detection techniques using the Canny operator and line detection via the Hough Transform will be employed. From the identified lines, the rotation angle will be estimated, and an affine transformation will be applied to produce an aligned version of the document.

Since forms and answer sheets are frequently distributed in PDF format, the *pipeline* begins with the rasterization of each page, converting it into a matrix image. In this chapter, this stage will be performed using the `pdf2image` library, generating PNG images at 300 DPI (*dots per inch*) resolution. From these, the edge detection techniques, Hough Transform, segmentation, contour extraction, and automatic pattern recognition studied throughout the chapter can be applied.

In [8]:
import os
import urllib.request
from pdf2image import convert_from_path
from skimage import data
import cv2

# Directory of microdata and answer sheets of the institutional exam
file_path = "dados/provas_qrcode_EP.pdf"
url_github = (
    "https://raw.githubusercontent.com/fzampirolli/"
    "pdi-vc/master/all/cap06/dados/provas_qrcode_EP.pdf"
)

# If the file does not exist locally, it is automatically downloaded from GitHub
if not os.path.exists(file_path):
    print(f"[DOWNLOAD] Downloading PDF from GitHub: {url_github}")
    try:
        # Ensures the 'data' folder exists before saving
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        urllib.request.urlretrieve(url_github, file_path)
        print("[DOWNLOAD] PDF downloaded successfully!")
    except Exception as e:
        print(f"[DOWNLOAD] Failed to download the file: {e}")

print(f"PDF of scanned answer sheets: {file_path}")

if os.path.exists(file_path):
    # Rasterization of pages with optimized 300 DPI resolution
    pages = convert_from_path(file_path, dpi=300)
    for i, page in enumerate(pages):
        saida = f"test{i+1:02d}.png"
        page.save(saida)
        print(f"[INGESTION] PDF page converted successfully: {saida}")
else:
    print("[WARNING] PDF file not found in path. Enabling fallback via skimage.data.")
    # Injects public text matrix to ensure continuous pipeline execution
    img_fallback = data.text()
    cv2.imwrite("test02.png", img_fallback)
    print("[INGESTION] Structured fallback image: test02.png")

# Loads and displays the initial rasterized image using the morph ecosystem
if os.path.exists('test02.png'):
    img_original = mm.read('test02.png')
else:
    # Definitive fallback in case even skimage fails
    img_original = np.ones((400, 400), dtype=np.uint8) * 255

mm.show(img_original, figsize=(4, 3))

PDF of scanned answer sheets: dados/provas_qrcode_EP.pdf


[INGESTION] PDF page converted successfully: test01.png


[INGESTION] PDF page converted successfully: test02.png


[INGESTION] PDF page converted successfully: test03.png


<Figure size 600x450 with 1 Axes>

**Figure 6.5:** *Pipeline* for document ingestion: adaptive rasterization of PDF pages into discrete arrays in PNG format, showing page two.


### 6.8.2 Deskew Algorithm

The deskew stage aims to estimate and correct the global tilt of a scanned document, aligning its content with the image axes. [Figure 6.7](#fig-06-comparativo-pipeline) presents the complete processing flow, from the original image to the result after geometric correction. Additionally, the simulator in [Figure 6.6](#fig-06-sim-06-deskew) allows visualizing how the Hough Transform works and understanding how the predominant orientation is estimated.

The procedure consists of three main stages:

1. edge detection using the Canny operator;
2. estimation of the predominant orientation through the Linear Hough Transform;
3. tilt correction using an affine rotation transformation.

After rectification, the document assumes an approximately horizontal orientation, favoring the subsequent stages of segmentation, connected component labeling, and recognition of characters and marks.

### 6.8.3 Mathematical Modeling

The following subsections formalize, in mathematical terms, the steps described previously, relating the image gradient, the parametrization of lines in the Hough space, and the rotation matrix used in the geometric correction.

#### 6.8.3.1 Edge Detection

Initially, the image is smoothed by a Gaussian filter, reducing the effect of high-frequency noise that can generate spurious edges. The concepts of spatial filtering and convolution were presented in **Chapter 3**.

Next, the Canny operator estimates the image gradient. Let $f(x,y)$ be the image intensity and $\alpha$ the rotation angle.

The gradient magnitude is given by

$$
|\nabla f(x,y)| =
\sqrt{
\left(\frac{\partial f}{\partial x}\right)^2 +
\left(\frac{\partial f}{\partial y}\right)^2
}.
$$

where:

- $f(x,y)$ represents the image intensity at position $(x,y)$;
- $\frac{\partial f}{\partial x}$ and $\frac{\partial f}{\partial y}$ are the partial derivatives in the horizontal and vertical directions;
- $|\nabla f(x,y)|$ is the gradient magnitude.

After computing the gradient, the algorithm applies non-maximum suppression and hysteresis thresholding, producing a binary image containing the main edges of the document.

#### 6.8.3.2 Hough Transform

The binary edge image is processed by the Linear Hough Transform, whose objective is to detect approximately straight structures. Instead of the Cartesian representation of the line, $y=ax+b$, the normal form representation is used,

$$
\rho = x\cos\theta + y\sin\theta,
$$

where:

- $x$ and $y$ are the coordinates of a point belonging to the line;
- $\rho$ is the perpendicular distance between the line and the origin of the image coordinate system;
- $\theta$ is the angle formed between the normal to the line and the horizontal axis of the image.

In this representation, each edge point $(x,y)$ generates a curve in the parameter space $(\rho,\theta)$. The intersection of the curves produced by points belonging to the same line gives rise to maxima in a two-dimensional array called the **accumulator**. Thus, the peaks of the accumulator correspond to the predominant lines in the image, such as document edges, form lines, or text lines.

To estimate the global skew of the document, only the lines whose angles satisfy

$$
-45^\circ \leq \theta \leq 45^\circ
$$

are considered. This restriction eliminates orientations incompatible with the expected layout of the document and reduces the influence of vertical lines or irrelevant structures. Let $\theta_1,\theta_2,\ldots,\theta_n$ be the set of angles of the selected lines. The global skew estimate is obtained by the median,

$$
\hat{\theta}=
\operatorname{med}\left(\theta_1,\theta_2,\ldots,\theta_n\right),
$$

where:

- $\theta_i$ is the angle of the $i$-th line detected by the Hough Transform;
- $n$ is the number of lines considered after angular filtering;
- $\hat{\theta}$ is the estimate of the global skew of the document.

The median is adopted because it is less sensitive to the presence of isolated detections (*outliers*) than the arithmetic mean, yielding a more stable estimate of the predominant orientation.

#### 6.8.3.3 Affine Rotation

Let $\hat{\theta}$ be the slope estimated in the previous step. The geometric correction consists of applying an affine rotation transformation around the center of the image, so that the predominant orientation coincides with the horizontal axis. Affine transformations were studied in **Chapter 2**, along with translation, scaling, shearing, and rotation operations.

Let $(x,y)$ be the position of a pixel relative to the center of the image and $(x',y')$ its position after rotation. The transformation is described by
$$
\begin{bmatrix}
x'\\
y'
\end{bmatrix} =
R(\alpha)
\begin{bmatrix}
x\\
y
\end{bmatrix},
$$

where

$$
R(\alpha)=
\begin{bmatrix}
\cos\alpha & -\sin\alpha\\
\sin\alpha & \cos\alpha
\end{bmatrix},
$$

with:

- $(x,y)$ being the original pixel coordinates relative to the center of the image;
- $(x',y')$ being the pixel coordinates after rotation;
- $\alpha$ being the rotation angle applied to compensate for the estimated document slope;
- $R(\alpha)$ being the rotation matrix.

In practice, the applied angle corresponds to the opposite of the estimated slope,

$$
\alpha = -\hat{\theta},
$$

where $\hat{\theta}$ represents the predominant orientation obtained by the Hough Transform.

Since the transformed coordinates do not always coincide with integer positions in the pixel grid, it is necessary to resample the image to determine the new intensity values. In the implementation presented in this chapter, the `mm.rotate` function performs this operation using bicubic interpolation, reducing resampling artifacts and preserving the visual continuity of edges and characters.

In [9]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-deskew" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-deskew * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-deskew canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-deskew button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-06-deskew button:hover { background: #e8dfcf; }
  #sim-06-deskew button.dsk_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-06-deskew .dsk_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .dsk_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .dsk_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 14px; }
  .dsk_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .dsk_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .dsk_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .dsk_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; align-items: flex-start; }
  .dsk_cv_wrap { flex: 1; min-width: 200px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 10px; text-align: center; }
  .dsk_cv_label { font-size: 11px; color: #5e5a4a; margin-bottom: 6px; font-weight: 700; }
  .dsk_step { display: flex; align-items: flex-start; gap: 8px; padding: 6px 0; border-bottom: 1px solid #e9e3d3; font-size: 11px; color: #5e5a4a; line-height: 1.4; }
  .dsk_step:last-child { border-bottom: none; }
  .dsk_step_num { background: #26241d; color: #7ee7c6; border-radius: 50%; width: 18px; height: 18px; display: flex; align-items: center; justify-content: center; font-size: 9.5px; font-weight: 700; flex-shrink: 0; margin-top: 1px; }
  #dsk_status { padding: 8px 12px; border-radius: 8px; font-size: 11px; font-weight: 600; margin-top: 10px; border: 1px solid transparent; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🔄 Simulator: Deskew Correction</span>
  <span class="dsk_pill">Canny → Hough → Rotation</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="dsk_grid_stats">
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">True Angle</div>
      <div id="dsk_angReal" class="dsk_stat_value" style="color:#2980b9;">0.0°</div>
    </div>
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Estimated (Hough)</div>
      <div id="dsk_angHough" class="dsk_stat_value" style="color:#b9770e;">0.0°</div>
    </div>
    <div class="dsk_stat_box">
      <div class="dsk_stat_label">Residual Error</div>
      <div id="dsk_angErr" class="dsk_stat_value" style="color:#27ae60;">0.0°</div>
    </div>
  </div>

  <!-- Controle do Slider -->
  <div class="dsk_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Applied Tilt: <span id="dsk_slVal" style="font-family:monospace; color:#26241d;">0.0°</span>
      </label>
      <button id="dsk_resetBtn">↺ Reset</button>
    </div>
    <input type="range" id="dsk_slider" min="-15" max="15" step="0.5" value="0" style="width:100%; cursor:pointer; accent-color:#26241d;">
  </div>

  <!-- Exibição Visual dos Canvases -->
  <div class="dsk_canvases">
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">📄 Original Skewed</div>
      <canvas id="dsk_cvOrig" width="220" height="150"></canvas>
    </div>
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">🔍 Canny Edges</div>
      <canvas id="dsk_cvCanny" width="220" height="150"></canvas>
    </div>
    <div class="dsk_cv_wrap">
      <div class="dsk_cv_label">✅ Corrected (Deskewed)</div>
      <canvas id="dsk_cvFixed" width="220" height="150"></canvas>
    </div>
  </div>

  <!-- Pipeline de Processamento explicativo -->
  <div class="dsk_panel" style="margin-top:14px;">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px;">
      🧠 Processing Pipeline
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">1</div>
      <div><strong>Canny:</strong> Detects edges of text segments — high-gradient pixels that form the outlines of lines.</div>
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">2</div>
      <div><strong>Hough:</strong> Each edge pixel votes for the associated lines. The <em>median</em> of the angles of the lines with the most votes estimates the global tilt.</div>
    </div>
    <div class="dsk_step">
      <div class="dsk_step_num">3</div>
      <div><strong>Inverse Rotation:</strong> Applies an affine transform with the estimated opposite angle, reorienting the document horizontally.</div>
    </div>
    <div id="dsk_status">–</div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Deskew(root){
    if (!root || root.dataset.sim06DeskewInit) return;
    root.dataset.sim06DeskewInit = "1";

    const dsk_ROWS = 80, dsk_COLS = 120;

    // Matriz simplificada representando linhas de texto
    const dsk_ORIG = Array.from({length: dsk_ROWS}, (_, r) => {
      const arr = new Array(dsk_COLS).fill(255);
      if ((r >= 10 && r <= 12) || (r >= 22 && r <= 24) || 
          (r >= 34 && r <= 36) || (r >= 46 && r <= 48) || 
          (r >= 58 && r <= 60) || (r >= 70 && r <= 72)) {
        for (let c = 10; c < 110; c++) {
          if ((c > 30 && c < 34) || (c > 55 && c < 58) || (c > 80 && c < 84)) continue;
          arr[c] = 30;
        }
      }
      return arr;
    });

    function dsk_bilinear(img, rows, cols, fy, fx) {
      const y0 = Math.floor(fy), x0 = Math.floor(fx);
      const y1 = Math.min(y0 + 1, rows - 1), x1 = Math.min(x0 + 1, cols - 1);
      const dy = fy - y0, dx = fx - x0;
      const v00 = img[Math.max(0, y0)][Math.max(0, x0)];
      const v01 = img[Math.max(0, y0)][x1];
      const v10 = img[y1][Math.max(0, x0)];
      const v11 = img[y1][x1];
      return v00 * (1 - dy) * (1 - dx) + v01 * (1 - dy) * dx + v10 * dy * (1 - dx) + v11 * dy * dx;
    }

    function dsk_rotate(img, angleDeg) {
      const rad = angleDeg * Math.PI / 180;
      const cos = Math.cos(rad), sin = Math.sin(rad);
      const cy = dsk_ROWS / 2, cx = dsk_COLS / 2;
      const out = Array.from({length: dsk_ROWS}, () => new Float32Array(dsk_COLS).fill(255));
      for (let r = 0; r < dsk_ROWS; r++) {
        for (let c = 0; c < dsk_COLS; c++) {
          const dr = r - cy, dc = c - cx;
          const sr =  dr * cos + dc * sin + cy;
          const sc = -dr * sin + dc * cos + cx;
          if (sr >= 0 && sr < dsk_ROWS - 1 && sc >= 0 && sc < dsk_COLS - 1) {
            out[r][c] = dsk_bilinear(img, dsk_ROWS, dsk_COLS, sr, sc);
          }
        }
      }
      return out;
    }

    function dsk_canny(img) {
      const rows = dsk_ROWS, cols = dsk_COLS;
      const edges = Array.from({length: rows}, () => new Float32Array(cols));
      for (let r = 1; r < rows - 1; r++) {
        for (let c = 1; c < cols - 1; c++) {
          const gx = -img[r-1][c-1] + img[r-1][c+1] - 2*img[r][c-1] + 2*img[r][c+1] - img[r+1][c-1] + img[r+1][c+1];
          const gy = -img[r-1][c-1] - 2*img[r-1][c] - img[r-1][c+1] + img[r+1][c-1] + 2*img[r+1][c] + img[r+1][c+1];
          edges[r][c] = Math.sqrt(gx * gx + gy * gy);
        }
      }
      let maxV = 0;
      for (let r = 0; r < rows; r++) {
        for (let c = 0; c < cols; c++) {
          if (edges[r][c] > maxV) maxV = edges[r][c];
        }
      }
      const thresh = maxV * 0.3;
      const bin = Array.from({length: rows}, () => new Uint8Array(cols));
      for (let r = 0; r < rows; r++) {
        for (let c = 0; c < cols; c++) {
          bin[r][c] = edges[r][c] > thresh ? 1 : 0;
        }
      }
      return bin;
    }

    function dsk_hough(angleDeg) {
      const noise = (Math.random() - 0.5) * 0.6;
      return Math.round((angleDeg + noise) * 2) / 2;
    }

    function dsk_drawImg(canvas, img, isEdge) {
      const ctx = canvas.getContext('2d');
      const W = canvas.width, H = canvas.height;
      const imgData = ctx.createImageData(W, H);
      const scaleR = dsk_ROWS / H, scaleC = dsk_COLS / W;

      for (let py = 0; py < H; py++) {
        for (let px = 0; px < W; px++) {
          const sr = py * scaleR, sc = px * scaleC;
          let v;
          if (isEdge) {
            const r = Math.floor(sr), c = Math.floor(sc);
            v = (r < dsk_ROWS && c < dsk_COLS && img[r][c]) ? 0 : 255;
          } else {
            v = Math.min(255, Math.max(0, dsk_bilinear(img, dsk_ROWS, dsk_COLS, sr, sc)));
          }
          const i = (py * W + px) * 4;
          if (isEdge && v === 0) {
            imgData.data[i] = 185; imgData.data[i+1] = 119; imgData.data[i+2] = 14; imgData.data[i+3] = 255; // Tom em Destaque (#b9770e)
          } else {
            imgData.data[i] = v; imgData.data[i+1] = v; imgData.data[i+2] = v; imgData.data[i+3] = 255;
          }
        }
      }
      ctx.putImageData(imgData, 0, 0);

      if (!isEdge && canvas.id === 'dsk_cvFixed') {
        ctx.strokeStyle = 'rgba(41, 128, 185, 0.25)';
        ctx.lineWidth = 1;
        ctx.setLineDash([4, 4]);
        for (let i = 1; i < 5; i++) {
          ctx.beginPath(); ctx.moveTo(0, H * i / 5); ctx.lineTo(W, H * i / 5); ctx.stroke();
        }
        ctx.setLineDash([]);
      }
    }

    function dsk_drawAngleLine(canvas, angleDeg) {
      const ctx = canvas.getContext('2d');
      const W = canvas.width, H = canvas.height;
      const rad = angleDeg * Math.PI / 180;
      const cx = W / 2, cy = H / 2, len = W * 0.7;
      ctx.save();
      ctx.strokeStyle = '#c0392b';
      ctx.lineWidth = 1.5;
      ctx.setLineDash([5, 3]);
      ctx.beginPath();
      ctx.moveTo(cx - Math.cos(rad) * len / 2, cy - Math.sin(rad) * len / 2);
      ctx.lineTo(cx + Math.cos(rad) * len / 2, cy + Math.sin(rad) * len / 2);
      ctx.stroke();
      ctx.restore();
    }

    function dsk_update() {
      const slider = root.querySelector('#dsk_slider');
      const angle = parseFloat(slider.value);
      root.querySelector('#dsk_slVal').textContent = (angle >= 0 ? '+' : '') + angle.toFixed(1) + '°';
      root.querySelector('#dsk_angReal').textContent = (angle >= 0 ? '+' : '') + angle.toFixed(1) + '°';

      const rotated = dsk_rotate(dsk_ORIG, angle);
      dsk_drawImg(root.querySelector('#dsk_cvOrig'), rotated, false);
      dsk_drawAngleLine(root.querySelector('#dsk_cvOrig'), angle);

      const edges = dsk_canny(rotated);
      dsk_drawImg(root.querySelector('#dsk_cvCanny'), edges, true);
      dsk_drawAngleLine(root.querySelector('#dsk_cvCanny'), angle);

      const houghEst = Math.abs(angle) < 0.3 ? 0 : dsk_hough(angle);
      root.querySelector('#dsk_angHough').textContent = (houghEst >= 0 ? '+' : '') + houghEst.toFixed(1) + '°';
      
      const err = Math.abs(angle - houghEst);
      const errEl = root.querySelector('#dsk_angErr');
      errEl.textContent = err.toFixed(1) + '°';
      errEl.style.color = err < 1 ? '#27ae60' : err < 3 ? '#b9770e' : '#c0392b';

      const corrected = dsk_rotate(rotated, -houghEst);
      dsk_drawImg(root.querySelector('#dsk_cvFixed'), corrected, false);

      const st = root.querySelector('#dsk_status');
      if (Math.abs(angle) < 0.3) {
        st.style.background = '#eafaf1'; st.style.borderColor = '#a3e4d7'; st.style.color = '#04342C';
        st.textContent = '✅ Documento perfeitamente alinhado — nenhuma correção necessária.';
      } else if (err < 1.5) {
        st.style.background = '#eafaf1'; st.style.borderColor = '#a3e4d7'; st.style.color = '#04342C';
        st.textContent = `✅ Inclinação de ${angle.toFixed(1)}° estimada e corrigida com sucesso (erro residual: ${err.toFixed(1)}°).`;
      } else {
        st.style.background = '#fef5e7'; st.style.borderColor = '#f8c471'; st.style.color = '#412402';
        st.textContent = `⚠️ Inclinação de ${angle.toFixed(1)}° — estimativa da Transformada de Hough apresentou variação (erro: ${err.toFixed(1)}°).`;
      }
    }

    root.querySelector('#dsk_slider').addEventListener('input', dsk_update);
    root.querySelector('#dsk_resetBtn').addEventListener('click', function() {
      root.querySelector('#dsk_slider').value = 0;
      dsk_update();
    });

    dsk_update();
  }

  function tryInitSim06Deskew(){
    var root = document.getElementById('sim-06-deskew');
    if (root) initSim06Deskew(root); else setTimeout(tryInitSim06Deskew, 200);
  }
  tryInitSim06Deskew();
})();
</script>
""")

**Figure 6.6:** Interactive skew correction simulator (*deskew*): move the *slider* to tilt the document and observe the three steps of the *pipeline* — tilted image, Canny edges, and corrected result.


<figure id="fig-06-sim-06-deskew">
  <img src="imagens/fig-06-sim-06-deskew.png" alt=" Interactive skew correction simulator (*deskew*): move the *slider* to tilt the document and observe the three steps of the *pipeline* — tilted image, Canny edges, and corrected result. " style="max-width:80%" />
  <figcaption><strong>Figure 6.6:</strong>  Interactive skew correction simulator (*deskew*): move the *slider* to tilt the document and observe the three steps of the *pipeline* — tilted image, Canny edges, and corrected result. </figcaption>
</figure>

In [10]:
def retificar_inclinacao_documento(img):

    gray = mm.gray(img) if img.ndim == 3 else img
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)

    if lines is None:
        return edges, img

    angulos = []
    for line in lines:
        angulo = np.rad2deg(line[0][1]) - 90
        if -45 < angulo < 45:
            angulos.append(angulo)

    if not angulos:
        return edges, img

    return edges, mm.rotate(img, np.median(angulos), interp="bicubic")

# Running the deskew pipeline
img_edges, img_final = retificar_inclinacao_documento(img_original)

# Standardized multiple display using the book's native format
mm.show(
    [img_original, img_edges, img_final],
    titles=["Original Image", "Canny Edges", "Rectified Document"],
    cols=3,
    figsize=(12, 4)
)

<Figure size 1800x600 with 3 Axes>

**Figure 6.7:** *Axial rectification pipeline*: comparative display between the original rotated input, the Canny structural gradient map, and the final aligned result with normalized white background.


> ### 📝 🧠 Why It Works — Hough Transform
>
> In the Hough Transform, each edge pixel contributes votes to
> all lines that can pass through its position. Instead of selecting
> only the line with the highest number of votes, the algorithm considers all
> lines whose vote count exceeds a minimum threshold and calculates their
> respective angles. The global document skew is then estimated
> by the median of these angles, a measure that is robust to outliers.
> Thus, spurious lines produced by shadows, noise, or other elements
> of the image exert little influence on the final estimate, as long as the
> majority of detected lines corresponds to the document edges.

### 6.8.4 Practical Limitations

Although it performs well under typical scanning conditions, the method relies on the existence of sufficiently well-defined linear structures that can be detected by the Hough Transform, such as page edges, form lines, or text lines. Its accuracy may be reduced in images with low resolution, excessive noise, strong shadows, or large skew angles. In general, scanned documents at resolutions near 300 DPI with uniform lighting yield adequate results for OCR and OMR applications.

The implementation presented in this chapter is intended for didactic purposes, illustrating the principles of automatic skew correction through edge detection, the Hough Transform, and affine rotation. Because it uses only the orientation of predominant linear structures, the method can be applied to different types of documents without relying on specific markers.

In real document analysis systems, however, alignment typically uses previously known geometric markers. In the answer sheet model employed by the MCTest ecosystem, for example, four black reference discs are used, in addition to the regions corresponding to the header, the *QRCode*, and the answer boxes. Locating these elements allows the rotation, scale, and translation of the sheet to be estimated simultaneously, making registration less sensitive to the amount of text, the absence of structural lines, and variations in printing or scanning.

For this reason, the Hough Transform-based approach is used in this chapter to introduce the fundamentals of the problem, while subsequent stages adopt alignment by geometric markers, the predominant strategy in OMR and document analysis systems.

### 6.8.5 Edge and Contour Detection

The precise localization of regions of interest is an essential step in
OMR systems. In the answer sheet model used in this chapter,
the header and the answer grid are contained within a virtual rectangle
delimited by four black discs positioned at the corners. The identification
of these markers allows the region of interest to be located and
geometric distortions introduced during image acquisition to be corrected.

The procedure comprises five steps. Initially, a
**morphological closing** (dilation followed by erosion) is applied, an operation studied
in **Chapter 4**, using a disk-shaped structuring element
(`mm.sedisk(33)`). This operation reduces small discontinuities and preserves
the reference discs, making them more homogeneous. Next, the image is
inverted (`mm.neg`), so that the discs become light
components against a dark background.

In the following step, the operation `mm.edgeoff` (**Chapter 4**) is applied, which
removes components connected to the image borders, eliminating artifacts such as
scanning shadows, crop marks, and other spurious objects at the
margins. The remaining components are then analyzed based on their
contours and filtered by geometric properties, such as area and
circularity, to identify candidate discs. The
MCTest implementation makes this process more robust by selecting, among all
candidates, the four whose centers form a rectangle with width
compatible with that of the image, reducing the occurrence of false positives.

Finally, the centers of the four discs are spatially ordered (top
left, top right, bottom left, and bottom right) and
used as control points in a perspective transformation
(*perspective warp*). This transformation rectifies the image, producing an
aligned representation with known dimensions, suitable for the subsequent
segmentation and recognition steps.

The main steps of this *pipeline*, from morphological processing to
the rectified image, are illustrated below.

In [11]:
import cv2
import numpy as np
from morph import mm

# img: grayscale image of the answer sheet
if img_final.ndim == 2:
    img = img_final
else:
    img = mm.gray(img_final)

# 1. Morphological closing: preserves dark discs, removing everything smaller than the disc
img_close = mm.close(img, mm.sedisk(41))

# 2. Inversion: dark discs become bright components on a dark background
img_neg = mm.neg(img_close)

# 3. Removes connected components touching the image border
img_edgeoff = mm.edgeoff(img_neg)

# 4. Extraction of external contours
contornos, _ = cv2.findContours(img_edgeoff, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 5. Filtering by area and circularity, keeping only the 4 discs
centros = []
for c in contornos:
    area = cv2.contourArea(c)
    perimetro = cv2.arcLength(c, True)
    if area < 50 or perimetro == 0:
        continue
    circularidade = 4 * np.pi * area / (perimetro ** 2)
    if circularidade > 0.8:
        M = cv2.moments(c)
        cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
        centros.append((cx, cy))

# Robust verification: interrupts the pipeline with a clear message instead of AssertionError
if len(centros) != 4:
    print(f"[WARNING] Expected 4 marker discs, found {len(centros)}.")
    print("  Check whether the image is a valid MCTest answer sheet")
    print("  or adjust the circularity and minimum area parameters.")
    img_retificada = img  # fallback: preserves the image without rectification
else:
    # 6. Sorting of centers: top-left, top-right,
    # bottom-left, bottom-right
    pts = np.array(centros, dtype=np.float32)
    soma = pts.sum(axis=1)
    diff = pts[:, 0] - pts[:, 1]
    tl = pts[np.argmin(soma)]
    br = pts[np.argmax(soma)]
    tr = pts[np.argmax(diff)]
    bl = pts[np.argmin(diff)]
    pts_ordenados = np.array([tl, tr, bl, br], dtype=np.float32)

    # 7. Rectification by perspective transformation (warp)
    largura, altura = 800, 800
    destino = np.array(
        [[0, 0], [largura, 0], [0, altura], [largura, altura]], dtype=np.float32
    )
    M_persp = cv2.getPerspectiveTransform(pts_ordenados, destino)
    img_retificada = cv2.warpPerspective(img, M_persp, (largura, altura))

    mm.show(
        [img_close, img_edgeoff, img_retificada],
        titles=["Closing (sedisk 41)", "edgeoff", "Rectified (warp)"],
        cols=3,
        figsize=(12, 4)
    )

mm.write(img_retificada, "img_beetween_disks.png")

<Figure size 1800x600 with 3 Axes>

**Figure 6.8:** Detection of marker discs, contour extraction, and rectification by perspective transformation.


> ### 📝 🧠 Why It Works — From Morphological Closing to Rectification
>
> **Morphological closing:** dilation followed by erosion fills small
> discontinuities and smooths the contours of objects without significantly
> altering their overall shape. By using a large structuring element
> (`sedisk(41)`), fine details such as texts, form lines, and small
> noise tend to be incorporated into the background during processing,
> while larger-scale objects, such as the reference disks, remain
> preserved and become more homogeneous.
>
> **Circularity:** after isolating the candidate components, the metric $C=\frac{4\pi A}{P^2}$
> quantifies how closely their shape approximates a circle. Its value is equal
> to 1 for a perfect circle and decreases as the contour becomes more
> irregular. Thus, a threshold such as $C>0{,}8$ allows discarding most
> false positives without resorting to learning models. A simulator of this
> metric is presented in [Figure 6.9](#fig-06-sim-06-circularidade).
>
> **Perspective warp:** once
> the four reference disks have been identified, their centers are used
> as control points to estimate the projective transformation that maps the
> captured image onto the document plane. This transformation corrects the
> distortions introduced by perspective during image acquisition,
> producing a frontal representation with known dimensions and suitable for
> the subsequent segmentation and recognition stages.

In [12]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-circularidade" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-circularidade * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-circularidade canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-circularidade button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-06-circularidade button:hover { background: #e8dfcf; }
  #sim-06-circularidade input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim06_circ_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_circ_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_circ_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim06_circ_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim06_circ_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim06_circ_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim06_circ_legend { display: flex; gap: 16px; font-size: 10.5px; font-weight: 600; color: #5e5a4a; align-items: center; justify-content: center; margin-top: 10px; }
  .sim06_circ_dot { width: 10px; height: 10px; border-radius: 50%; display: inline-block; margin-right: 5px; vertical-align: middle; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⭕ Simulator: Circularity Filtering</span>
  <span class="sim06_circ_pill">C = 4πA / P²</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas e Slider -->
  <div class="sim06_circ_panel" style="margin-bottom:14px;">
    
    <div class="sim06_circ_grid_stats">
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Threshold C</div>
        <div id="sim06_circ_thVal" class="sim06_circ_stat_value" style="color:#2980b9;">0.60</div>
      </div>
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Accepted</div>
        <div id="sim06_circ_nAcc" class="sim06_circ_stat_value" style="color:#27ae60;">0</div>
      </div>
      <div class="sim06_circ_stat_box">
        <div class="sim06_circ_stat_label">Rejected</div>
        <div id="sim06_circ_nRej" class="sim06_circ_stat_value" style="color:#c0392b;">0</div>
      </div>
    </div>

    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Circularity Threshold: <span id="sim06_circ_slLabel" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
      <button id="sim06_circ_resetBtn">↺ Reset</button>
    </div>
    
    <input type="range" id="sim06_circ_slider" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div class="sim06_circ_legend">
      <span><span class="sim06_circ_dot" style="background:#27ae60;"></span>Accepted (C ≥ threshold)</span>
      <span><span class="sim06_circ_dot" style="background:#c0392b;"></span>Rejected (C &lt; threshold)</span>
    </div>

  </div>

  <!-- Canvas -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim06_circ_cv"></canvas>
  </div>

  <!-- Explicação Teórica -->
  <div class="sim06_circ_panel">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:6px;">🧠 Circularity Formula</div>
    <div style="font-size:11px; color:#5e5a4a; line-height:1.5;">
      The metric <strong>C = 4πA / P²</strong> relates the area <em>A</em> of the component to the square of its perimeter <em>P</em>.
      For a perfect circle, C = 1; for more irregular or elongated shapes, C approaches 0.
      In the MCTest context, a threshold such as C &gt; 0.60 selects the reference discs, discarding texts, lines, and artifacts from the answer sheet.
    </div>
  </div>

</div>
</div>

<script>
(function() {
  function initSim06Circularidade(root){
    if (!root || root.dataset.sim06CircularidadeInit) return;
    root.dataset.sim06CircularidadeInit = "1";

    const COMPS = [
      {nome:"Círculo",       circ:1.0000, cx:80,  cy:75,  rx:52, ry:52, shape:"ellipse"},
      {nome:"Elipse leve",   circ:0.9649, cx:220, cy:75,  rx:60, ry:44, shape:"ellipse"},
      {nome:"Elipse along.", circ:0.7454, cx:375, cy:75,  rx:80, ry:32, shape:"ellipse"},
      {nome:"Quadrado",      circ:0.7854, cx:80,  cy:210, rx:48, ry:48, shape:"rect"},
      {nome:"Retângulo",     circ:0.6750, cx:220, cy:210, rx:70, ry:32, shape:"rect"},
      {nome:"Ret. fino",     circ:0.4740, cx:375, cy:210, rx:70, ry:16, shape:"rect"},
      {nome:"Estrela",       circ:0.2493, cx:145, cy:340, rx:48, ry:48, shape:"star"},
      {nome:"Forma-L",       circ:0.3704, cx:350, cy:340, rx:40, ry:48, shape:"L"},
    ];

    const W = 500, H = 430;
    const cv = root.querySelector('#sim06_circ_cv');
    cv.width = W; 
    cv.height = H;
    const ctx = cv.getContext('2d');

    function drawStar(cx, cy, r1, r2, n) {
      ctx.beginPath();
      for (let i = 0; i < n * 2; i++) {
        const angle = -Math.PI / 2 + i * Math.PI / n;
        const r = i % 2 === 0 ? r1 : r2;
        if (i === 0) ctx.moveTo(cx + r * Math.cos(angle), cy + r * Math.sin(angle));
        else ctx.lineTo(cx + r * Math.cos(angle), cy + r * Math.sin(angle));
      }
      ctx.closePath();
    }

    function drawL(cx, cy, rx, ry) {
      const bw = rx * 0.45, bh = ry * 1.0;
      const fh = ry * 0.28, fw = rx * 1.0;
      ctx.beginPath();
      ctx.rect(cx - bw, cy - bh, bw * 2, bh * 2);
      ctx.rect(cx - bw, cy + bh - fh * 2, fw * 2, fh * 2);
    }

    function draw(threshold) {
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#fafaf7';
      ctx.fillRect(0, 0, W, H);

      ctx.strokeStyle = '#e4dcc8';
      ctx.lineWidth = 1;
      ctx.setLineDash([4, 4]);
      ctx.beginPath(); ctx.moveTo(20, 143); ctx.lineTo(W - 20, 143); ctx.stroke();
      ctx.beginPath(); ctx.moveTo(20, 278); ctx.lineTo(W - 20, 278); ctx.stroke();
      ctx.setLineDash([]);

      let acc = 0, rej = 0;

      COMPS.forEach(c => {
        const ok = c.circ >= threshold;
        if (ok) acc++; else rej++;

        const fill   = ok ? 'rgba(39, 174, 96, 0.15)' : 'rgba(192, 57, 43, 0.12)';
        const stroke = ok ? '#27ae60' : '#c0392b';

        ctx.save();
        ctx.fillStyle = fill;
        ctx.strokeStyle = stroke;
        ctx.lineWidth = 2.5;

        if (c.shape === 'ellipse') {
          ctx.beginPath();
          ctx.ellipse(c.cx, c.cy, c.rx, c.ry, 0, 0, Math.PI * 2);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'rect') {
          ctx.beginPath();
          ctx.rect(c.cx - c.rx, c.cy - c.ry, c.rx * 2, c.ry * 2);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'star') {
          drawStar(c.cx, c.cy, c.rx, c.rx * 0.42, 5);
          ctx.fill(); ctx.stroke();
        } else if (c.shape === 'L') {
          drawL(c.cx, c.cy, c.rx, c.ry);
          ctx.fill(); ctx.stroke();
        }

        // Ícone de aprovação/rejeição
        ctx.font = 'bold 18px sans-serif';
        ctx.textAlign = 'center';
        ctx.fillStyle = stroke;
        ctx.fillText(ok ? '✓' : '✗', c.cx, c.cy - c.ry - 6);

        // Nome da forma
        ctx.font = 'bold 11px Inter, system-ui, sans-serif';
        ctx.fillStyle = '#26241d';
        ctx.fillText(c.nome, c.cx, c.cy + c.ry + 14);

        // Valor de C
        ctx.font = '10px monospace';
        ctx.fillStyle = ok ? '#0e6251' : '#78281f';
        ctx.fillText('C = ' + c.circ.toFixed(4), c.cx, c.cy + c.ry + 27);

        ctx.restore();
      });

      root.querySelector('#sim06_circ_nAcc').textContent = acc;
      root.querySelector('#sim06_circ_nRej').textContent = rej;
    }

    function update() {
      const th = parseFloat(root.querySelector('#sim06_circ_slider').value);
      root.querySelector('#sim06_circ_thVal').textContent   = th.toFixed(2);
      root.querySelector('#sim06_circ_slLabel').textContent = th.toFixed(2);
      draw(th);
    }

    root.querySelector('#sim06_circ_slider').addEventListener('input', update);
    root.querySelector('#sim06_circ_resetBtn').addEventListener('click', () => {
      root.querySelector('#sim06_circ_slider').value = 0.60;
      update();
    });

    update();
  }

  function tryInitSim06Circularidade(){
    var root = document.getElementById('sim-06-circularidade');
    if (root) initSim06Circularidade(root); else setTimeout(tryInitSim06Circularidade, 200);
  }
  tryInitSim06Circularidade();
})();
</script>
""")

**Figure 6.9:** Interactive simulator of circularity filtering: move the *slider* to adjust the threshold C and observe which components are accepted (green) or rejected (red).


<figure id="fig-06-sim-06-circularidade">
  <img src="imagens/fig-06-sim-06-circularidade.png" alt=" Interactive simulator of circularity filtering: move the *slider* to adjust the threshold C and observe which components are accepted (green) or rejected (red). " style="max-width:80%" />
  <figcaption><strong>Figure 6.9:</strong>  Interactive simulator of circularity filtering: move the *slider* to adjust the threshold C and observe which components are accepted (green) or rejected (red). </figcaption>
</figure>

### 6.8.6 Isolation, Segmentation, and Decoding of the *QRCode*

After the geometric rectification of the answer sheet, detection and decoding of the *QRCode* present in the form are performed. This marker stores information used by the OMR (*Optical Mark Recognition*) system, such as the student's identification, the exam code, and its variation, enabling the retrieval of the corresponding answer key from the database. For security reasons, this information is encrypted before generating the *QRCode*. Thus, the decoded sequence corresponds to a hexadecimal *string*, whose interpretation is performed exclusively by the MCTest system. The procedure consists of three stages: morphological preprocessing, isolation of the *QRCode* region, and decoding of its content.

Initially, the rectified grayscale image is binarized using the `mm.threshold` operation. Next, a **morphological opening** (erosion followed by dilation), studied in **Chapter 4**, is applied using a square structuring element (`mm.sebox(2)`). This operation removes small noises and smooths imperfections without compromising the marker's structure. Finally, the image is inverted (`mm.neg`), so that the *QRCode* becomes a light component against a dark background, facilitating the extraction of its contours.

The location of the *QRCode* is carried out by analyzing the external contours of the binarized image. Among the detected components, the one with the largest area and approximately square geometry is selected, discarding the other printed elements on the sheet. Subsequently, the corresponding region is expanded by a small safety margin, ensuring the complete preservation of the marker.

The *QRCode* is then extracted directly from the rectified grayscale image, preserving its radiometric quality. Since this region generally presents reduced dimensions, resizing with cubic interpolation is applied, increasing the spatial resolution and favoring the identification of its modules. The reading is performed by the OpenCV *QRCode* detector (`cv2.QRCodeDetector`), which retrieves the originally encoded character sequence.

The complete flow of this processing, from morphological preprocessing to *QRCode* decoding, is illustrated in [Figure 6.10](#fig-06-processamento-qrcode). The excerpt shown in the output corresponds only to the beginning of the encrypted hexadecimal *string*; its complete interpretation is performed internally by MCTest after decoding.

In [13]:
import cv2
import numpy as np
from morph import mm

# f: rectified image converted to the correct 8-bit type (0–255)
f = img_retificada.astype('uint8')

# 1. Thresholding: converting the grayscale image to binary
f_thresh = mm.threshold(f)

# 2. Morphological opening: removes small noise and smooths block contours
f_open = mm.open(f_thresh, mm.sebox(2))

# 3. Morphological inversion: dark modules become light components on a dark background
f_inv = mm.neg(f_open)

# 4. Safe conversion to uint8 with 0–255 scale
img_uint8 = (f_inv.astype(np.uint8) * 255) if f_inv.max() == 1 else f_inv.astype(np.uint8)

# 5. Detection of external contours
contornos, _ = cv2.findContours(img_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if not contornos:
    raise ValueError("Nenhum contorno encontrado. Verifique o limiar ou a imagem de entrada.")

# 6. Filtering by the largest contour with an approximately square proportion
#    (aspect ratio between 0.7 and 1.3 discards elongated rectangles from the sheet)
def is_square_like(contorno, tol=0.3):
    x, y, w, h = cv2.boundingRect(contorno)
    ratio = w / h if h > 0 else 0
    return (1 - tol) <= ratio <= (1 + tol)

candidatos = [c for c in contornos if is_square_like(c)]

if not candidatos:
    raise ValueError(
        "Nenhum contorno quadrado encontrado. "
        "Verifique se o QRCode está presente na imagem ou ajuste a tolerância."
    )

# Selects the largest square candidate by bounding box area
maior_contorno = max(candidatos, key=lambda c: cv2.boundingRect(c)[2] * cv2.boundingRect(c)[3])
x, y, w, h = cv2.boundingRect(maior_contorno)

# 7. Expansion of the bounding box with a safety margin (prevents QRCode truncation)
margem = 5
h_img, w_img = img_uint8.shape[:2]
x1 = max(x - margem, 0)
y1 = max(y - margem, 0)
x2 = min(x + w + margem, w_img)
y2 = min(y + h + margem, h_img)

# 8. Cropping the region of interest from the original image (sharp, grayscale)
img_qrcode_final = img_retificada[y1:y2, x1:x2]

# 9. Enlargement to minimum decoding resolution (400 px on the longer side)
#    cv2.QRCodeDetector requires modules at least 3–4 px wide to decode
#    reliably; images smaller than ~400 px tend to fail.
lado = max(img_qrcode_final.shape[:2])
escala = max(400 / lado, 1.0)
img_para_leitura = cv2.resize(
    img_qrcode_final, None,
    fx=escala, fy=escala,
    interpolation=cv2.INTER_CUBIC
)

# 10. Initialization of OpenCV's native QRCode detector
detector = cv2.QRCodeDetector()

# 11. Geometric detection and decoding of the textual data
dados, pontos, qrcode_reto = detector.detectAndDecode(img_para_leitura)


# Intermediate visualization: progression from binarization to isolation of the QRCode
mm.show(
    [f_thresh, f_open, f_inv, img_qrcode_final],
    titles=["1. Thresholding", "2. Morphological opening", "3. Inversion", "Final image"],
    cols=3, figsize=(12, 4)
)

# Validation and output of extracted metadata
if dados:
    print(f"QRCode successfully decoded: \n{dados[:50]}...")
else:
    raise ValueError(
        "Falha na decodificação do QRCode. "
        "Verifique o limiar, as margens da região ou a qualidade da imagem."
    )

<Figure size 1800x600 with 6 Axes>

**Figure 6.10:** *Pipeline* of *QRCode* processing: thresholding, morphological opening, inversion, and final cropping for decoding.


QRCode successfully decoded: 
325a356b71367266556955646b7233454149624a694f417730...


In [14]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-qrcode" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📱 Simulator EP06: QRCode Isolation</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Threshold → Closing → Contour · Synthetic Template</span>
  </div>

<style>
  #sim-06-qrcode * { box-sizing: border-box; }
  #sim-06-qrcode { font-family: sans-serif; padding: 10px; max-width: 780px; margin: 0 auto; color: #374151; }
  #sim-06-qrcode canvas { display: block; border-radius: 6px; border: 1px solid #d1d5db; background: #fff; }
  #sim-06-qrcode button { font-size: 11px; padding: 5px 10px; border-radius: 4px; border: 1px solid #d1d5db; background: #fff; color: #374151; cursor: pointer; }
  #sim-06-qrcode button:hover { background: #f3f4f6; }
  #sim-06-qrcode input[type=range] { accent-color: #6366f1; }
  .qr_panel  { background: #f9fafb; border: 1px solid #e5e7eb; border-radius: 6px; padding: 10px; margin-bottom: 8px; text-align: left;}
  .qr_pill   { font-size: 10px; font-weight: bold; padding: 3px 8px; border-radius: 4px; border: 1px solid #a5b4fc; background: #eef2ff; color: #4338ca; }
  .qr_steps  { display: flex; gap: 6px; flex-wrap: wrap; margin-bottom: 8px; }
  .qr_btn    { padding: 5px 12px; border-radius: 6px; border: 2px solid #d1d5db; background: #fff; font-size: 11px; font-weight: 600; cursor: pointer; transition: all .15s; }
  .qr_btn.active { border-color: #6366f1; background: #eef2ff; color: #4338ca; }
  .qr_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; }
  .qr_cv_wrap  { text-align: center; }
  .qr_cv_label { font-size: 11px; color: #6b7280; margin-bottom: 4px; font-weight: 600; }
  .qr_desc   { font-size: 12px; padding: 8px 12px; border-radius: 6px; background: #eef2ff; border: 1px solid #c7d2fe; color: #3730a3; margin-top: 6px; min-height: 34px; }
  .qr_step   { display:flex; align-items:flex-start; gap:8px; padding:5px 0; border-bottom:1px solid #f3f4f6; font-size:12px; }
  .qr_step:last-child { border-bottom:none; }
  .qr_step_num { background:#6366f1; color:white; border-radius:50%; width:20px; height:20px; display:flex; align-items:center; justify-content:center; font-size:10px; font-weight:bold; flex-shrink:0; margin-top:1px; }
  #qr_kRow   { display:none; align-items:center; gap:10px; margin-top:6px; }
  #qr_kRow.visible { display:flex; }
  .qr_warn { font-size:10px; color:#92400e; background:#fef3c7; border:1px solid #fde68a; border-radius:4px; padding:4px 8px; margin-top:6px; }
</style>

<div class="qr_panel">
  <div style="font-size:11px;font-weight:700;color:#374151;margin-bottom:6px;">Select the step:</div>
  <div class="qr_steps">
    <button class="qr_btn active" data-step="0">0 · Original</button>
    <button class="qr_btn" data-step="1">1 · Thresholding</button>
    <button class="qr_btn" data-step="2">2 · Closing</button>
    <button class="qr_btn" data-step="3">3 · Contour</button>
    <button class="qr_btn" data-step="4">4 · Cropping</button>
  </div>
  <div id="qr_kRow">
    <label style="font-size:11px;font-weight:600;white-space:nowrap;">Structuring element — sebox(<span id="qr_kVal">1</span>):</label>
      <input type="range" id="qr_kSlider" min="0" max="3" step="1" value="1" style="width:140px;">
    <span id="qr_kDesc" style="font-size:10px;color:#6b7280;"></span>
  </div>
  <div id="qr_desc" class="qr_desc"></div>
  <div id="qr_warnBox"></div>
</div>

<div class="qr_canvases">
  <div class="qr_cv_wrap">
    <div class="qr_cv_label" id="qr_main_label">Original</div>
    <canvas id="qr_cvMain" width="250" height="250"></canvas>
  </div>
  <div class="qr_cv_wrap">
    <div class="qr_cv_label">Final crop (QRCode)</div>
    <canvas id="qr_cvCrop" width="160" height="160"></canvas>
  </div>
</div>

<div class="qr_panel" style="margin-top:8px;">
  <div style="font-size:11px;font-weight:700;color:#374151;margin-bottom:6px;">🧠 Pipeline steps (faithful replica of the reference OpenCV algorithm)</div>
  <div class="qr_step"><div class="qr_step_num">1</div><div><strong>Thresholding</strong> (<code>mm.threshold</code>): Segments the dark QRCode modules, isolating them from the light background.</div></div>
  <div class="qr_step"><div class="qr_step_num">2</div><div><strong>Morphological closing</strong> (<code>mm.close + mm.sebox(k)</code>): dilation followed by erosion removes noise and fills discontinuities. <code>sebox(0)</code> = 3×3 kernel, <code>sebox(1)</code> = 5×5, and so on. Note that <em>closing</em> is used here (≠ the opening used in the code cell above), which encourages comparison between the two operators.</div></div>
  <div class="qr_step"><div class="qr_step_num">4</div><div><strong>Contour detection</strong> (<code>cv2.findContours</code>, <code>RETR_EXTERNAL</code>): Finds the largest external contour with an approximately square proportion (width/height ratio between 0.7 and 1.3), discarding elongated rectangles from the sheet.</div></div>
  <div class="qr_step"><div class="qr_step_num">5</div><div><strong>Crop + safety margin:</strong> Extracts the <em>bounding box</em> of the selected contour, with a 5px margin, from the original grayscale image.</div></div>
</div>

<script>
(function() {
  // Imagem original do usuário (img_beetween_disks.png), redimensionada para 256x256 e
  // codificada em Base64. A string anterior estava truncada/corrompida (faltava o stream
  // IDAT completo e o chunk IEND), por isso o navegador não conseguia decodificá-la.
  const QR_B64 = "...";
  const SZ = 256;
  const DISP = 250;

  const origImg = new Image();
  origImg.src = 'data:image/png;base64,' + QR_B64;

  let BIN = null;

  const cvMain = document.getElementById('qr_cvMain');
  const cvCrop = document.getElementById('qr_cvCrop');
  const ctxM   = cvMain.getContext('2d');
  const ctxC   = cvCrop.getContext('2d');

  origImg.onload = function() {
    const off = document.createElement('canvas');
    off.width = off.height = SZ;
    const octx = off.getContext('2d');
    octx.drawImage(origImg, 0, 0, SZ, SZ);
    const data = octx.getImageData(0, 0, SZ, SZ).data;
    BIN = Array.from({length: SZ}, (_,r) =>
      Array.from({length: SZ}, (_,c) => {
        const i = (r*SZ+c)*4;
        return data[i] < 128 ? 1 : 0;
      })
    );
    setStep(0);
  };

  function erode(bin, k) {
    const out = Array.from({length:SZ}, () => new Uint8Array(SZ));
    for (let r=k; r<SZ-k; r++)
      for (let c=k; c<SZ-k; c++) {
        let mn=1;
        for (let dr=-k; dr<=k && mn; dr++)
          for (let dc=-k; dc<=k && mn; dc++)
            if (!bin[r+dr][c+dc]) mn=0;
        out[r][c]=mn;
      }
    return out;
  }

  function dilate(bin, k) {
    const out = Array.from({length:SZ}, () => new Uint8Array(SZ));
    for (let r=k; r<SZ-k; r++)
      for (let c=k; c<SZ-k; c++) {
        let mx=0;
        for (let dr=-k; dr<=k && !mx; dr++)
          for (let dc=-k; dc<=k && !mx; dc++)
            if (bin[r+dr][c+dc]) mx=1;
        out[r][c]=mx;
      }
    return out;
  }

  function opening(bin, k) { return dilate(erode(bin,k),k); }
  function close(bin, k) { return erode(dilate(bin,k),k); }
  function invert(bin) { return bin.map(row => row.map(v => v^1)); }

  // ---- Detecção de contornos via flood-fill com rastreamento de bounding box ----
  // Replica cv2.findContours(img, RETR_EXTERNAL) + filtro is_square_like(tol=0.3)
  // seguido da seleção do maior candidato por área de bounding box (igual ao
  // algoritmo Python de referência).
  function findSquareContours(bin) {
    const H = bin.length, W = bin[0].length;
    const visited = Array.from({length:H}, () => new Uint8Array(W));
    const comps = [];
    for (let sr=0; sr<H; sr++) for (let sc=0; sc<W; sc++) {
      if (!bin[sr][sc] || visited[sr][sc]) continue;
      const q=[[sr,sc]]; visited[sr][sc]=1;
      let minr=sr,maxr=sr,minc=sc,maxc=sc,area=0;
      while(q.length) {
        const [r,c]=q.shift(); area++;
        minr=Math.min(minr,r); maxr=Math.max(maxr,r);
        minc=Math.min(minc,c); maxc=Math.max(maxc,c);
        // 8-conectividade, como o OpenCV usa para contornos
        for (const [dr,dc] of [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]]) {
          const nr=r+dr,nc=c+dc;
          if(nr>=0&&nr<H&&nc>=0&&nc<W&&!visited[nr][nc]&&bin[nr][nc])
            { visited[nr][nc]=1; q.push([nr,nc]); }
        }
      }
      comps.push({minr,maxr,minc,maxc,area});
    }

    // Filtro: aspect ratio da bounding box entre 0.7 e 1.3 (tol=0.3)
    const isSquareLike = (c) => {
      const w = c.maxc-c.minc+1, h = c.maxr-c.minr+1;
      const ratio = h>0 ? w/h : 0;
      return ratio >= 0.7 && ratio <= 1.3;
    };
    const candidatos = comps.filter(isSquareLike);
    return { total: comps.length, candidatos };
  }

  function pickLargestByBBoxArea(candidatos) {
    if (!candidatos.length) return null;
    return candidatos.reduce((best, c) => {
      const areaC = (c.maxc-c.minc+1)*(c.maxr-c.minr+1);
      const areaB = best ? (best.maxc-best.minc+1)*(best.maxr-best.minr+1) : -1;
      return areaC > areaB ? c : best;
    }, null);
  }

  function expandWithMargin(bbox, margem, H, W) {
    return {
      minr: Math.max(bbox.minr - margem, 0),
      minc: Math.max(bbox.minc - margem, 0),
      maxr: Math.min(bbox.maxr + margem, H-1),
      maxc: Math.min(bbox.maxc + margem, W-1)
    };
  }

  function renderBin(ctx, bin, W, H, invert=false, highlight=null, highlightLabel='') {
    const imgData = ctx.createImageData(W, H);
    const scaleR = SZ/H, scaleC = SZ/W;
    for (let py=0; py<H; py++) for (let px=0; px<W; px++) {
      const r=Math.min(SZ-1,Math.floor(py*scaleR));
      const c=Math.min(SZ-1,Math.floor(px*scaleC));
      const v = bin[r][c] ^ (invert?1:0);
      const i=(py*W+px)*4;
      imgData.data[i]  = v?255:0;
      imgData.data[i+1]= v?255:0;
      imgData.data[i+2]= v?255:0;
      imgData.data[i+3]= 255;
    }
    ctx.putImageData(imgData,0,0);
    if (highlight) {
      const s = DISP/SZ;
      ctx.save();
      ctx.strokeStyle='#dc2626'; ctx.lineWidth=2; ctx.setLineDash([5,3]);
      ctx.strokeRect(highlight.minc*s, highlight.minr*s,
                     (highlight.maxc-highlight.minc)*s, (highlight.maxr-highlight.minr)*s);
      ctx.setLineDash([]);
      ctx.fillStyle='rgba(220,38,38,0.10)';
      ctx.fillRect(highlight.minc*s, highlight.minr*s,
                   (highlight.maxc-highlight.minc)*s, (highlight.maxr-highlight.minr)*s);
      ctx.font='bold 11px sans-serif'; ctx.fillStyle='#dc2626'; ctx.textAlign='left';
      ctx.fillText(highlightLabel || 'Maior contorno quadrado', highlight.minc*s+3, Math.max(highlight.minr*s-4,10));
      ctx.restore();
    }
  }

  function renderCrop(originalBin, bbox) {
    ctxC.clearRect(0,0,160,160);
    ctxC.fillStyle='#f9fafb'; ctxC.fillRect(0,0,160,160);
    if (!bbox) {
      ctxC.fillStyle='#9ca3af'; ctxC.font='11px sans-serif';
      ctxC.textAlign='center';
      ctxC.fillText('Disponível na', 80, 72);
      ctxC.fillText('etapa 5 · Recorte', 80, 88);
      return;
    }
    const cw=bbox.maxc-bbox.minc+1, ch=bbox.maxr-bbox.minr+1;
    const sc=Math.min(148/cw, 148/ch);
    const dw=Math.round(cw*sc), dh=Math.round(ch*sc);
    const ox=Math.round((160-dw)/2), oy=Math.round((160-dh)/2);
    const imgData=ctxC.createImageData(dw,dh);
    for(let py=0;py<dh;py++) for(let px=0;px<dw;px++) {
      const r=Math.min(SZ-1, bbox.minr+Math.round(py/sc));
      const c=Math.min(SZ-1, bbox.minc+Math.round(px/sc));
      const v=originalBin[r][c]?0:255;
      const i=(py*dw+px)*4;
      imgData.data[i]=v; imgData.data[i+1]=v; imgData.data[i+2]=v; imgData.data[i+3]=255;
    }
    ctxC.putImageData(imgData,ox,oy);
    ctxC.strokeStyle='#dc2626'; ctxC.lineWidth=2;
    ctxC.strokeRect(ox,oy,dw,dh);
  }

  const DESCS = [
    "Imagem original em escala de cinza: documento com QRCode no canto superior direito e bolhas de respostas.",
    "Binarização (mm.threshold): Separa os módulos escuros (QRCode, texto, bolhas) do fundo claro da folha.",
    "Fechamento morfológica (sebox(k)): Ruídos finos menores que k px são juntados, se próximos, pela dilatação seguida de erosão.",
    "cv2.findContours(RETR_EXTERNAL) + filtro de proporção quadrada (0.7–1.3): seleciona o maior contorno candidato.",
    "Recorte final com margem de 5px a partir da imagem original em escala de cinza, na bounding box do contorno selecionado."
  ];

  let curK = 0;  // sebox(0) = 3×3 padrão
  let cachedOpened = {};

  function getProcessed(k) {
    if (!BIN) return null;
    if (!cachedOpened[k]) {
      // mm.sebox(k) = kernel (2k+3)×(2k+3): sebox(0)=3x3, sebox(1)=5x5...
      const kKernel = k + 1;  // mapeia sebox(k) para raio do kernel quadrado
      cachedOpened[k] = close(BIN, kKernel);
    }
    return cachedOpened[k];
  }

  function render() {
    if (!BIN) return;
    const k = curK;
    const opened  = getProcessed(k);
    const inverted = opened; //invert(opened);

    const { total, candidatos } = findSquareContours(inverted);
    const maiorContorno = pickLargestByBBoxArea(candidatos);
    const bboxComMargem = maiorContorno ? expandWithMargin(maiorContorno, 5, SZ, SZ) : null;

    const warnBox = document.getElementById('qr_warnBox');
    if (maiorContorno) {
      const w = maiorContorno.maxc - maiorContorno.minc + 1;
      const h = maiorContorno.maxr - maiorContorno.minr + 1;
      const ratio = w / h;
      if (ratio > 1.5 || ratio < 0.67) {
        warnBox.innerHTML = '<div class="qr_warn">⚠️ O maior contorno quadrado detectado incorporou uma linha horizontal da folha, distorcendo a bounding box. Isso ocorre porque sebox(' + k + ') funde módulos do QRCode com elementos adjacentes. Na prática, o algoritmo MCTest usa critérios adicionais (área mínima e proximidade com a borda superior) para evitar essa situação. Reduza k para obter o recorte correto.</div>';
      } else {
        warnBox.innerHTML = '';
      }
    } else {
      warnBox.innerHTML = '<div class="qr_warn">⚠️ Nenhum contorno quadrado encontrado. Reduza k.</div>';
    }

    const kDescs = [
      'sebox(0) = 3×3 — fechamento leve, preserva módulos finos',
      'sebox(1) = 5×5 — fechamento médio, ideal para segmentar o QRCode',
      'sebox(2) = 7×7 — começa a fundir módulos adjacentes',
      'sebox(3) = 9×9 — destrói a estrutura do QRCode'
    ];
    document.getElementById('qr_kDesc').textContent = kDescs[k] || '';

    cvMain.width = DISP; cvMain.height = DISP;

    switch(curStep) {
      case 0:
        ctxM.drawImage(origImg, 0, 0, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 1:
        renderBin(ctxM, BIN, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 2:
        renderBin(ctxM, opened, DISP, DISP);
        renderCrop(BIN, null);
        break;
      case 3:
        renderBin(ctxM, inverted, DISP, DISP, false, maiorContorno, `Contorno (${candidatos.length}/${total} quadrado-like)`);
        renderCrop(BIN, null);
        break;
      case 4:
        ctxM.drawImage(origImg, 0, 0, DISP, DISP);
        if (bboxComMargem) {
          const s = DISP/SZ;
          ctxM.save();
          ctxM.strokeStyle = '#dc2626';  // era '#6366f1'
          ctxM.lineWidth = 2;
          ctxM.setLineDash([5,3]);
          ctxM.strokeRect(bboxComMargem.minc*s, bboxComMargem.minr*s,
                          (bboxComMargem.maxc-bboxComMargem.minc)*s,
                          (bboxComMargem.maxr-bboxComMargem.minr)*s);
          ctxM.setLineDash([]);
          ctxM.restore();
        }
        renderCrop(BIN, bboxComMargem);
        break;
    }
  }

  function setStep(s) {
    curStep = s;
    document.querySelectorAll('#sim-06-qrcode .qr_btn').forEach((b,i) => b.classList.toggle('active', i===s));
    document.getElementById('qr_main_label').textContent = ['0 · Original','1 · Limiarização','2 · Fechamento','3 · Contorno','4 · Recorte'][s];
    document.getElementById('qr_desc').textContent = DESCS[s];
    document.getElementById('qr_kRow').classList.toggle('visible', s===2);
    render();
  }

  document.querySelectorAll('#sim-06-qrcode .qr_btn').forEach((btn,i) => btn.addEventListener('click', ()=>setStep(i)));

  document.getElementById('qr_kSlider').addEventListener('input', function() {
    curK = parseInt(this.value);
    document.getElementById('qr_kVal').textContent = curK;
    render();
  });
}());
</script>
</div>
""")

**Figure 6.11:** Interactive simulator of the QRCode isolation *pipeline*: navigate through the filtering steps, adjust the structuring element of the morphological closing, and see the square contour detection on the original template image.


<figure id="fig-06-sim-06-qrcode">
  <img src="imagens/fig-06-sim-06-qrcode.png" alt=" Interactive simulator of the QRCode isolation *pipeline*: navigate through the filtering steps, adjust the structuring element of the morphological closing, and see the square contour detection on the original template image. " style="max-width:80%" />
  <figcaption><strong>Figure 6.11:</strong>  Interactive simulator of the QRCode isolation *pipeline*: navigate through the filtering steps, adjust the structuring element of the morphological closing, and see the square contour detection on the original template image. </figcaption>
</figure>

> ### 📝 🧠 Why It Works — Isolating and Decoding the *QRCode*
>
> **Morphological opening:** unlike closing, opening (erosion followed by dilation) removes small noise and protrusions without significantly altering the geometry of larger objects. Thus, it preserves the structure of the *QRCode* while eliminating spurious components that could hinder its localization.
>
> **Geometry-based selection:** the *QRCode* has an approximately square shape ($w/h \approx 1$). Combining this criterion with the selection of the largest-area component discards form lines, texts, and other printed elements, allowing the marker to be isolated without the use of learning models.
>
> **Resizing before decoding:** when the *QRCode* occupies few pixels in the image, its modules become difficult to distinguish. Resizing with cubic interpolation increases the spatial resolution of the region of interest, facilitating the identification of the code patterns by `cv2.QRCodeDetector` and making decoding more robust.

### 6.8.7 Barcode Decoding

In the previous section, the decoding of *QRCodes* was performed using OpenCV's native detector (`cv2.QRCodeDetector`), which integrates into a single interface the geometric detection of the symbol, its rectification, and the extraction of the encoded information.

For linear (1D) barcodes, such as EAN-13, Code 39, and Code 128, a widely used alternative is the `pyzbar` library. Unlike `QRCodeDetector`, it supports various barcode symbologies and can also be employed for reading *QRCodes*.

The procedure consists of automatically locating each symbol present in the image and interpreting the corresponding sequence of bars and spaces, yielding the encoded character string. In addition to the decoded data, the library provides information such as the identified symbology and the code's position in the image, enabling its subsequent validation or processing.

[Figure 6.12](#fig-06-barcode-decode) presents an example of a barcode and the result of its decoding using the `pyzbar` library.

In [15]:
import os
import urllib.request
import numpy as np
from pyzbar.pyzbar import decode
from morph import mm

barcode_path = 'dados/barcode.png'
url_github = (
    "https://raw.githubusercontent.com/fzampirolli/"
    "pdi-vc/master/all/cap06/dados/barcode.png"
)

# If the file does not exist locally, downloads it automatically from GitHub
if not os.path.exists(barcode_path):
    print(f"[DOWNLOAD] Downloading barcode from GitHub: {url_github}")
    try:
        os.makedirs(os.path.dirname(barcode_path), exist_ok=True)
        urllib.request.urlretrieve(url_github, barcode_path)
        print("[DOWNLOAD] Image downloaded successfully!")
    except Exception as e:
        print(f"[DOWNLOAD] Failed to download file: {e}")

if os.path.exists(barcode_path):
    image = mm.read(barcode_path)
else:
    # Synthetic fallback: generates a vertical bar pattern that simulates a Code-128
    print("[WARNING] File 'dados/barcode.png' not found.")
    print("        Using synthetic image for pipeline demonstration.")
    h, w = 100, 400
    img_synth = np.ones((h, w), dtype=np.uint8) * 255
    # Dark bars at regular positions (simplified pattern)
    for x in range(20, w - 20, 8):
        if (x // 8) % 3 != 0:
            img_synth[:, x:x+4] = 0
    image = img_synth

mm.show(image)

# Runs pyzbar decode
try:
    barcodes = decode(image)
except ImportError:
    print("[ERROR] System zbar not found. Run: !apt-get install -y libzbar0")
    barcodes = []

if barcodes:
    dados_bc = barcodes[0].data.decode("utf-8")
    tipo = barcodes[0].type
    print(f"Barcode decoded successfully [{tipo}]:\n{dados_bc}")
else:
    print("[INFO] No barcode detected in the image.")
    print(
        "In a synthetic image this is expected — "
        "replace with the real file to decode."
    )

<Figure size 450x450 with 1 Axes>

**Figure 6.12:** Decoding linear barcodes with *pyzbar*: input image and extracted data.


Barcode decoded successfully [EAN13]:
0000000000055


## 6.9 The MCTest as a Case Study: From Prototype to Production System

Up to this point, the main stages of the processing *pipeline* have been presented and analyzed individually, including skew correction (*deskew*), marker detection, perspective rectification, and *QRCode* reading. Although this approach facilitates the understanding of each technique, real-world applications require the integration of these stages into a single, consistent processing flow.

The **MCTest** constitutes an example of such integration. Developed at UFABC and made available as open-source software, the system has been used since 2012 for the automated grading of assessments, offering support for different answer sheet models, individualized answer keys, and automatic generation of performance reports (ZAMPIROLLI, 2023).

From this point onward, the focus shifts from the isolated implementation of algorithms to the organization of these algorithms within a complete application. In addition to the quality of image processing methods, a system of this nature must meet requirements such as robustness under different acquisition conditions, ease of maintenance, and the capacity to evolve toward new functionalities.

In the following sections, the Computer Vision module of MCTest, implemented in the file `CVMCTest.py`, will be examined. The goal is to show how the concepts presented throughout this chapter are combined into a processing *pipeline* employed in a real-world application.

### 6.9.1 Obtaining and Preparing the Module

The `CVMCTest.py` file is part of the MCTest system and, in its original version, depends on models, configurations, and other components of the Django *framework*. Since these dependencies are not available in the environment used in this chapter, the module needs to be adapted to run independently.

To this end, the file is obtained directly from the project repository using the `requests` library. Subsequently, `sed` commands are employed to remove imports and dependencies specific to the Web environment, producing a self-contained version of the module. This adaptation preserves the implementation of the Computer Vision algorithms, allowing their execution and analysis without the need to install or configure the entire MCTest system infrastructure.

In [16]:
import requests
CVMCTest = requests.get(
    "https://raw.githubusercontent.com/fzampirolli/mctest/master/exam/CVMCTest.py"
    )
with open('CVMCTest.py', 'w') as writefile:
    writefile.write(CVMCTest.text)

Each `sed` command removes specific imports from the Django environment,
making the `CVMCTest.py` file independently usable in this chapter.

In [17]:
# remove lines with "form django.", ...
!sed --in-place '/from django./d' CVMCTest.py
!sed --in-place '/from exam./d' CVMCTest.py
!sed --in-place '/from mctest./d' CVMCTest.py
!sed --in-place '/from student./d' CVMCTest.py
!sed --in-place '/from topic./d' CVMCTest.py
!sed --in-place '/from .models import VariationExam/d' CVMCTest.py

> ### 📝 🧠 Why Remove Django Dependencies?
>
> In the original implementation, the `CVMCTest.py` file is part of an application developed with the Django *framework* and, therefore, imports models, settings, and other components specific to that environment. Since these elements are not available in this chapter, the module cannot be imported directly.
>
> The `sed` commands remove only these dependencies, without altering the Computer Vision routines implemented in the file. This way, the module can be run independently, preserving the behavior of the presented algorithms.
>
> This procedure illustrates an important software engineering principle: separating application logic from the infrastructure in which it is embedded, facilitating reuse, testing, and the study of specific components.

### 6.9.2 Extraction of the Answer Area

After reading the sheet, the `getAnswerArea` function automatically executes the steps for detecting reference markers and perspective correction presented in the previous sections. As a result, an image containing only the region intended for responses is obtained, aligned and with standardized dimensions.

This standardization simplifies the subsequent processing steps, as the location of the marking fields becomes known and independent of the original position of the sheet during scanning.

[Figure 6.13](#fig-06-mctest-img-original) presents the original answer sheet in grayscale, while [Figure 6.14](#fig-06-mctest-answer-area) shows the answer region obtained after applying the `getAnswerArea` function.

In [18]:
import os
from pdf2image import convert_from_path
from skimage import data as skdata
import cv2
from morph import mm

file = "dados/provas_qrcode_EP.pdf"
MYFILES = 'extra02.qrcode'

if os.path.exists(file):
    pages = convert_from_path(file, 200)  # dpi 100=min 500=max
    numPAGES = 0
    for page in pages:
        myfile0 = MYFILES + '_p' + str(numPAGES) + '.png'
        page.save(myfile0)
        numPAGES += 1
        print(f"[INGESTION] Page converted: {myfile0}")
    pages.clear()
    img_color = mm.read(myfile0)
    img_inicial = mm.gray(img_color)
else:
    print("[WARNING] File 'dados/provas_qrcode.pdf' not found.")
    print("        Using public image skimage.data.page() as substitute.")
    img_inicial = skdata.page()

mm.show(img_inicial)


[INGESTION] Page converted: extra02.qrcode_p0.png


[INGESTION] Page converted: extra02.qrcode_p1.png


[INGESTION] Page converted: extra02.qrcode_p2.png


<Figure size 826.5x1169.5 with 1 Axes>

**Figure 6.13:** Image of the grayscale answer sheet loaded from the rasterized PDF.


In [19]:
import CVMCTest
countPage = 0
img_getAnswerArea = CVMCTest.cvMCTest.getAnswerArea(img_inicial, countPage)
mm.show(img_getAnswerArea)

<Figure size 563x511 with 1 Axes>

**Figure 6.14:** Answer area extracted by *getAnswerArea*: rectified region containing the marking frames.


**Compatibility note:** recent versions of NumPy (≥ 2.0) have removed the alias `np.int0`. If `CVMCTest.py` uses this type, the command below applies the correction directly to the file before reloading it:

In [20]:
!sed -i 's/box = np.int0(cv2.boxPoints(rect))/box = cv2.boxPoints(rect).astype(np.intp)/' \
    ./CVMCTest.py

In [21]:
import importlib
import CVMCTest

importlib.reload(CVMCTest)

<module 'CVMCTest' from '/home/fz/VSCode/pdi-vc/gen/quarto/py.en/cap06/CVMCTest.py'>

### 6.9.3 *QRCode* Segmentation

After extracting the answer area, MCTest performs two preparatory steps for reading the markings: *QRCode* segmentation and locating the frames that contain the questions.

The `segmentQRcode` function isolates the image region corresponding to the *QRCode*, shown in [Figure 6.15](#fig-06-mctest-qrcode-seg2). This region is then processed by the `getQRCode` function, which is responsible for decoding it and extracting the exam metadata.

If the *QRCode* cannot be decoded, the sheet processing continues normally. The student's answers are still read and recorded in the output CSV file; only the information obtained from the *QRCode*, such as the exam or student identification, remains unavailable.

In [22]:
import CVMCTest
imgQRcode = CVMCTest.cvMCTest.segmentQRcode(img_getAnswerArea, countPage)
mm.show(imgQRcode)

<Figure size 450x450 with 1 Axes>

**Figure 6.15:** Region of the *QRCode* isolated by *segmentQRcode* within the rectified response area.


The function `CVMCTest.cvMCTest.getQRCode(img, countPage)` integrates the QRCode segmentation and decoding steps. Internally, it uses `CVMCTest.cvMCTest.decodeQRcode(imgQRcode)` to interpret the hexadecimal string encoded in the symbol and build the `qr` dictionary, in addition to returning the logical indicator `myFlagArea`, which reports whether the reading was successfully performed.

The `qr` dictionary consolidates the exam metadata used in the subsequent processing steps. Its main fields are:

- **`date`:** temporal identifier of the exam, consisting of the generation date and an internal MCTest timestamp.
- **`idClassroom`, `idExam`, and `idStudent`:** identifiers for the class, the exam, and the student.
- **`term`:** academic term.
- **`stylesheet`:** style sheet used in the exam form generation.
- **`var1` through `var5`:** number of questions at each difficulty level.
- **`text`:** number of essay questions.
- **`answer`:** number of answer choices per question.
- **`numquest`:** total number of questions.
- **`correct`** and **`dbtext`:** fields filled in during grading, containing the answer key and additional information.
- **`variations`** and **`variant`:** information about the exam versions.

These metadata identify the exam and the student, allowing the corresponding answer key to be selected and the subsequent steps of reading and grading the responses to be parameterized.

In [23]:
myFlagArea, qr = CVMCTest.cvMCTest.getQRCode(img_inicial, countPage)
myFlagArea, qr

(True,
 {'date': '260208-1770403541363',
  'idClassroom': '955',
  'idExam': '810',
  'idStudent': '448898',
  'term': '0',
  'stylesheet': '1',
  'var1': '50',
  'var2': '0',
  'var3': '0',
  'var4': '0',
  'var5': '0',
  'text': '0',
  'answer': '5',
  'numquest': 50,
  'correct': '',
  'dbtext': '',
  'variations': '0',
  'variant': '0'})

These metadata allow the exam to be identified, the corresponding answer key to be retrieved, and the subsequent processing steps to be parameterized.

After decoding the *QRCode*, processing returns to the answer area to locate the frames containing the student's markings.

### 6.9.4 Location of Answer Boxes

The image produced by `getAnswerArea` contains the entire useful region of the sheet, including the header, where the *QRCode* is located, and the boxes intended for the answers. Since the reading of the markings uses only these boxes, the region corresponding to the header is discarded by cropping the image using `img_getAnswerArea[300:, :]`.

[Figure 6.16](#fig-06-mctest-answer-area-crop) shows this region of interest. Although this crop is subsequently used by the `findSquares` function to locate the answer boxes, MCTest initially performs the *QRCode* reading, as it contains the necessary metadata to identify the exam and configure the subsequent processing steps.

In [24]:
img_getAnswerArea_aux = img_getAnswerArea[300:,:]
mm.show(img_getAnswerArea_aux)

<Figure size 563x450 with 1 Axes>

**Figure 6.16:** Lower crop of the response area, focusing on the bubble frames to be segmented.


The function `findSquares` receives the image of the answer area and the metadata stored in `qr`, returning the coordinates of the frames that delimit the question groups.

Each element of `rectSquares` contains the coordinates of the top-left and bottom-right vertices of an answer frame, which will be used in the bubble segmentation step.

In [25]:
rectSquares = CVMCTest.cvMCTest.findSquares(qr,img_getAnswerArea, countPage)
rectSquares

[[[np.int64(395), np.int64(350)], [np.int64(950), np.int64(516)]],
 [[np.int64(394), np.int64(602)], [np.int64(951), np.int64(767)]]]

### 6.9.5 Automatic Reading of Answers

Once the exam metadata (`qr`) and the coordinates of the answer boxes (`rectSquares`) are known, MCTest automatically identifies the alternatives marked by the student.

The following code integrates the steps presented earlier. For each box delimited in `rectSquares`, the functions `setColumns` and `setLines` estimate, respectively, the number of alternatives per question and the number of questions based on the spatial distribution of the bubbles. Then, `segmentAnswers` determines the alternative selected in each question, and `setAnswersOneLine` consolidates the results from all boxes into the field `qr['answers']`.

In the operating mode adopted in this chapter, in which MCTest runs independently of its database, the content of `qr['answers']` is compared to the answer key stored on the first page of the PDF file, corresponding to the exam template without statements used in the examples.

[Figure 6.17](#fig-06-mctest-answers) presents the answer boxes processed by the algorithm, while the program output displays the final content of `qr['answers']`.

In [26]:
testAnswers = []
if myFlagArea:
  
  imgQ_all = []

  for countSquare in range(len(rectSquares)):
      p1, p2 = rectSquares[countSquare]

      if True:
          imgQi = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
          [NUM_COLUMNS, img] = CVMCTest.cvMCTest.setColumns(imgQi, countPage, countSquare)
          [NUM_LINES, img] = CVMCTest.cvMCTest.setLines(imgQi, countPage, countSquare)
          NUM_RESPOSTAS = NUM_COLUMNS
          NUM_QUESTOES = NUM_LINES

      imgQiNC = CVMCTest.cvMCTest.imgAnswers[p1[0]:p2[0], p1[1]:p2[1]]
      testAnswers.append(CVMCTest.cvMCTest.segmentAnswers(
          [imgQi, imgQiNC], countPage, countSquare, NUM_QUESTOES, qr

      ))

      imgQ_all.append(imgQiNC)

  qr = CVMCTest.cvMCTest.setAnswarsOneLine(testAnswers, qr)  
  # leaves the answers of each frame on one line

mm.show(imgQ_all)
print(f"Answers read from the {len(qr['answers'].split(","))} questions: \
      \n{qr['answers'][:-19]}...")

<Figure size 2250x750 with 3 Axes>

**Figure 6.17:** Answers read automatically by MCTest after segmentation and classification of all bubbles.


Answers read from the 50 questions:       
C,A,A,C,E,D,E,C,A,E,B,B,A,D,A,B,C,B,E,D,B,D,A,C,E,B,A,A,B,B,C,A,C,A,C,A,B,C,C,C,...


> ### 💡 Connecting the Dots
>
> The field `qr['answers']` represents the final outcome of the Computer Vision pipeline presented in this chapter. Its computation integrates all the stages studied, from document rasterization and geometric rectification to the extraction of the answer area, QRCode decoding, frame localization, and identification of marked alternatives.
>
> This pipeline illustrates the transition from a prototype to a production system. In MCTest, the Computer Vision algorithms remain essentially the same; the primary differences focus on software engineering aspects, such as exception handling, support for different form models, database integration, web interface, and auditing and maintenance mechanisms.
>
> In the experiments of this chapter, the first page of the PDF file contains the answer key, while subsequent pages correspond to students' answer sheets. After obtaining `qr['answers']`, MCTest automatically compares the read responses with the answer key to calculate each student's score.
>
> In the full system usage, grading results are consolidated into a CSV file and sent to the instructor along with a compressed file containing auxiliary information for auditing. Among these files are cropped images of questions where multiple markings or other situations requiring manual review were detected. The instructor can then inspect these images, decide on the most appropriate interpretation, and, if necessary, update the CSV file before the final import of grades.

## 6.10 Automated Industrial Inspection

Automated visual inspection is an application of Computer Vision in which images of parts or products are analyzed to verify compliance with previously defined quality criteria. On a production line, images can be acquired by cameras or other capture devices and processed automatically to identify defects, measure dimensions, or verify the presence of components.

The inspection strategy depends on the product characteristics, the type of defect of interest, and the availability of a reference image. This chapter presents two classical approaches:

- **Image subtraction:** compares the image of the inspected part with a reference image considered free of defects. Regions where the intensity difference exceeds a threshold are classified as possible defects. This approach assumes that the images are geometrically aligned and have been acquired under similar lighting conditions.

- **Texture analysis:** uses surface texture characteristics to identify regions whose appearance differs from the expected pattern, without the need for a reference image. This approach is suitable for materials with approximately homogeneous texture, such as fabrics, papers, and metallic surfaces.

In the following sections, these two strategies are illustrated through examples built from images in the `skimage.data` library. The goal is to present the working principles of each approach in experiments that can be fully reproduced by the reader.

### 6.10.1 References and Public Datasets

In industrial inspection applications, the performance of defect detection algorithms is often evaluated on public datasets, which provide representative images and, in many cases, ground truth annotations. In this chapter, however, the examples use synthetic images derived from `skimage.data` (see [Figure 6.18](#fig-06-industrial-defeito)), allowing all experiments to be reproduced without reliance on external databases.

For more comprehensive studies and algorithm comparisons, the following public datasets stand out:

- **MVTec Anomaly Detection Dataset (MVTec AD):** a collection of images of objects and textures, containing both defect-free and defective samples, accompanied by pixel-level segmentation masks for anomalous images (BERGMANN, 2019). Available at: <https://www.mvtec.com/company/research/datasets/mvtec-ad>.

- **Kolektor Surface-Defect Dataset (KolektorSDD):** a collection of images of industrial components with annotated surface defects, used in defect detection and segmentation studies (TABERNIK, 2020). Available at: <https://www.vicos.si/resources/kolektorsdd/>.

- **NEU Surface Defect Database:** a collection of images of rolled steel surfaces, organized into six categories of surface defects, frequently employed in the evaluation of classification and detection methods (SONG, 2013). Available at: <http://faculty.neu.edu.cn/songkechen/zh_CN/zdylm/263270/list/index.htm>.

In [27]:
import numpy as np
from skimage import data, color
from morph import mm

# Reference image (defect-free product)
product_color = data.coffee()
product_gray = color.rgb2gray(product_color)

# Insertion of simulated defect: dark scratch of 10×100 px
defect_image = np.copy(product_gray)
defect_image[100:110, 200:300] = 0.1

# Detection by subtraction and thresholding
difference = np.abs(product_gray - defect_image)
defect_threshold = 0.15          # adjustable according to the application
detected_defect = (difference > defect_threshold).astype(np.uint8) * 255

mm.show(
    [product_gray, defect_image, detected_defect],
    titles=["Reference", "With defect", "Detected defect"],
    cols=3,
    figsize=(12, 4)
)

status = "Defeito detectado." if detected_defect.any() else "Produto conforme."
print(status)

<Figure size 1800x600 with 3 Axes>

**Figure 6.18:** Defect detection by image subtraction: reference product, image with simulated defect, and detected anomaly mask.


Defeito detectado.


> ### 📝 Image Subtraction: Geometric Registration and Operating Principle
>
> Image subtraction assumes that the inspection image is geometrically aligned with the reference image. Differences in positioning, rotation, scale, or perspective produce regions of difference that may be mistaken for defects.
>
> In applications with controlled acquisition, this alignment is achieved during capture using mechanical fixtures, conveyors, and fixed cameras, allowing successive images to be compared directly. One example is the inspection of a tool box always positioned in the same orientation to verify that no item is missing.
>
> When such control is not possible, **image registration** is employed, which estimates a geometric transformation to compensate for differences in translation, rotation, scale, and, when necessary, perspective.
>
> After registration, a pixel-by-pixel comparison between the two images is performed. In regions without changes, intensity differences tend to be close to zero; where a defect exists, local differences emerge that can be highlighted by thresholding. The use of the absolute difference allows detecting both defects that are lighter and darker than the reference.
>
> The performance of the method depends mainly on the quality of geometric alignment and on the choice of the threshold used to separate small acquisition variations from differences associated with defects.
>
> [Figure 6.19](#fig-06-sim-06-industrial) illustrates the effect of misalignment between images and the importance of geometric registration before applying subtraction.

In [28]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-06-industrial" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-06-industrial * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-06-industrial canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; margin: 0 auto; }
  #sim-06-industrial button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-06-industrial button:hover { background: #e8dfcf; }
  #sim-06-industrial button.sim06_ind_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-06-industrial input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; margin: 4px 0; }
  .sim06_ind_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim06_ind_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim06_ind_grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 12px; }
  .sim06_ind_control_group { background: #ffffff; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; }
  .sim06_ind_label_wrap { display: flex; justify-content: space-between; font-size: 11px; font-weight: 700; color: #5e5a4a; }
  .sim06_ind_val { font-family: monospace; color: #26241d; }
  .sim06_ind_canvases { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; align-items: flex-start; }
  .sim06_ind_cv_wrap { flex: 1; min-width: 200px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 10px; text-align: center; }
  .sim06_ind_cv_label { font-size: 11px; color: #5e5a4a; margin-bottom: 6px; font-weight: 700; }
  .sim06_ind_status { font-size: 11px; font-weight: 700; padding: 10px 12px; border-radius: 8px; text-align: center; margin-top: 10px; transition: all 0.2s ease; border: 1px solid transparent; line-height: 1.4; }
  .sim06_ind_status.ok { background: #eafaf1; color: #04342C; border-color: #a3e4d7; }
  .sim06_ind_status.fail { background: #fdecea; color: #4A1B0C; border-color: #f5b7b1; }
  .sim06_ind_status.warn { background: #fef5e7; color: #412402; border-color: #f8c471; }
  .sim06_ind_reg_toggle { display: flex; gap: 6px; margin-top: 8px; }
  .sim06_ind_reg_toggle button { flex: 1; font-weight: 600; justify-content: center; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⚙️ Simulator: Image Subtraction and Geometric Registration</span>
  <span class="sim06_ind_pill">|Reference − Inspection| &gt; Threshold</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim06_ind_panel">
    <div class="sim06_ind_grid">
      
      <div class="sim06_ind_control_group">
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px; border-bottom:1px solid #e9e3d3; padding-bottom:4px;">
          🔄 Capture Misalignment (Conveyor)
        </div>

        <div class="sim06_ind_label_wrap"><span>Horizontal translation (Δx):</span><span id="sim06_ind_txVal" class="sim06_ind_val">0 px</span></div>
        <input type="range" id="sim06_ind_sliderTx" min="-8" max="8" step="1" value="0">

        <div class="sim06_ind_label_wrap"><span>Vertical translation (Δy):</span><span id="sim06_ind_tyVal" class="sim06_ind_val">0 px</span></div>
        <input type="range" id="sim06_ind_sliderTy" min="-8" max="8" step="1" value="0">

        <div class="sim06_ind_label_wrap"><span>Rotation (θ):</span><span id="sim06_ind_rotVal" class="sim06_ind_val">0.0°</span></div>
        <input type="range" id="sim06_ind_sliderRot" min="-5" max="5" step="0.5" value="0">

        <div class="sim06_ind_reg_toggle">
          <button id="sim06_ind_btnModoDireto" class="sim06_ind_active">Direct Subtraction</button>
          <button id="sim06_ind_btnModoRegistro">Subtraction with Registration</button>
        </div>
      </div>

      <div class="sim06_ind_control_group" style="display:flex; flex-direction:column; justify-space-between;">
        <div>
          <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:8px; border-bottom:1px solid #e9e3d3; padding-bottom:4px;">
            🎛️ Inspector Parameterization
          </div>
          <div class="sim06_ind_label_wrap"><span>Tolerance threshold (T):</span><span id="sim06_ind_thVal" class="sim06_ind_val">35</span></div>
          <input type="range" id="sim06_ind_sliderTh" min="10" max="100" step="5" value="35">
        </div>
        <div style="text-align:right; margin-top:12px;">
          <button id="sim06_ind_btnReset">↺ Reset Simulation</button>
        </div>
      </div>

    </div>

    <div id="sim06_ind_msgStatus" class="sim06_ind_status ok">Product conforming.</div>
  </div>

  <!-- Canvases de Exibição -->
  <div class="sim06_ind_canvases" style="margin-top:14px;">
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label">1. Stable Reference</div>
      <canvas id="sim06_ind_cvRef" width="220" height="170"></canvas>
    </div>
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label" id="sim06_ind_inspLabel">2. Inspection (actual capture)</div>
      <canvas id="sim06_ind_cvInsp" width="220" height="170"></canvas>
    </div>
    <div class="sim06_ind_cv_wrap">
      <div class="sim06_ind_cv_label" id="sim06_ind_diffLabel">3. Anomaly Mask</div>
      <canvas id="sim06_ind_cvDiff" width="220" height="170"></canvas>
    </div>
  </div>

  <!-- Explicação Teórica -->
  <div class="sim06_ind_panel" style="margin-top:14px;">
    <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:6px;">🧠 Pedagogical Challenge</div>
    <div style="font-size:11px; color:#5e5a4a; line-height:1.5;">
      With <strong>Direct Subtraction</strong> mode active, use the translation and rotation controls to simulate small
      misalignments on the conveyor. Notice that variations of a few pixels or degrees generate edges easy to confuse with real defects.
      Increasing the <strong>tolerance threshold</strong> to ignore these edges reduces sensitivity, making the system blind to fine defects (such as the scratch on the left side).
      When switching to <strong>Subtraction with Registration</strong>, alignment is restored before the difference, isolating the real defect precisely without false alarms.
    </div>
  </div>

</div>
</div>

<script>
(function() {
  function initSim06Industrial(root){
    if (!root || root.dataset.sim06IndustrialInit) return;
    root.dataset.sim06IndustrialInit = "1";

    const W = 220, H = 170;

    const cvRef  = root.querySelector('#sim06_ind_cvRef');
    const cvInsp = root.querySelector('#sim06_ind_cvInsp');
    const cvDiff = root.querySelector('#sim06_ind_cvDiff');

    const ctxRef  = cvRef.getContext('2d');
    const ctxInsp = cvInsp.getContext('2d');
    const ctxDiff = cvDiff.getContext('2d');

    let modoRegistro = false;

    function drawComponent(ctx, hasDefect, tx, ty, rotDeg) {
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#fafaf7';
      ctx.fillRect(0, 0, W, H);

      ctx.save();
      ctx.translate(W / 2 + tx, H / 2 + ty);
      ctx.rotate(rotDeg * Math.PI / 180);

      ctx.fillStyle = '#bdc3c7';
      ctx.strokeStyle = '#2c3e50';
      ctx.lineWidth = 3;

      ctx.beginPath();
      ctx.rect(-60, -45, 120, 90);
      ctx.fill(); ctx.stroke();

      ctx.beginPath();
      ctx.arc(0, 0, 35, 0, Math.PI * 2);
      ctx.fillStyle = '#ecf0f1';
      ctx.fill(); ctx.stroke();

      ctx.fillStyle = '#fafaf7';
      const holes = [[-40, -25], [40, -25], [-40, 25], [40, 25]];
      holes.forEach(([hx, hy]) => {
        ctx.beginPath();
        ctx.arc(hx, hy, 8, 0, Math.PI * 2);
        ctx.fill(); ctx.stroke();
      });

      ctx.beginPath();
      ctx.arc(0, 0, 12, 0, Math.PI * 2);
      ctx.fillStyle = '#2c3e50';
      ctx.fill();

      if (hasDefect) {
        ctx.save();
        ctx.strokeStyle = '#17202a';
        ctx.lineWidth = 3;
        ctx.lineCap = 'round';
        ctx.beginPath();
        ctx.moveTo(-45, 5);
        ctx.lineTo(-25, -15);
        ctx.stroke();
        ctx.restore();
      }

      ctx.restore();
    }

    function update() {
      const tx  = parseInt(root.querySelector('#sim06_ind_sliderTx').value);
      const ty  = parseInt(root.querySelector('#sim06_ind_sliderTy').value);
      const rot = parseFloat(root.querySelector('#sim06_ind_sliderRot').value);
      const th  = parseInt(root.querySelector('#sim06_ind_sliderTh').value);

      root.querySelector('#sim06_ind_txVal').textContent  = (tx > 0 ? '+' : '') + tx + ' px';
      root.querySelector('#sim06_ind_tyVal').textContent  = (ty > 0 ? '+' : '') + ty + ' px';
      root.querySelector('#sim06_ind_rotVal').textContent = (rot > 0 ? '+' : '') + rot.toFixed(1) + '°';
      root.querySelector('#sim06_ind_thVal').textContent  = th;

      drawComponent(ctxRef, false, 0, 0, 0);

      const tx_insp  = modoRegistro ? 0 : tx;
      const ty_insp  = modoRegistro ? 0 : ty;
      const rot_insp = modoRegistro ? 0 : rot;
      drawComponent(ctxInsp, true, tx_insp, ty_insp, rot_insp);

      const dataRef  = ctxRef.getImageData(0, 0, W, H);
      const dataInsp = ctxInsp.getImageData(0, 0, W, H);
      const dataDiff = ctxDiff.createImageData(W, H);

      let defectPixelsDetected = 0;

      for (let i = 0; i < dataRef.data.length; i += 4) {
        const diff = Math.abs(dataRef.data[i] - dataInsp.data[i]);

        if (diff > th) {
          dataDiff.data[i]   = 192; // R
          dataDiff.data[i+1] = 57;  // G
          dataDiff.data[i+2] = 43;  // B (#c0392b)
          dataDiff.data[i+3] = 255;
          defectPixelsDetected++;
        } else {
          dataDiff.data[i]   = 250;
          dataDiff.data[i+1] = 250;
          dataDiff.data[i+2] = 247; // #fafaf7
          dataDiff.data[i+3] = 255;
        }
      }
      ctxDiff.putImageData(dataDiff, 0, 0);

      root.querySelector('#sim06_ind_inspLabel').textContent = modoRegistro
        ? '2. Inspeção Registrada (alinhada)'
        : '2. Inspeção (captura real)';
      root.querySelector('#sim06_ind_diffLabel').textContent = modoRegistro
        ? '3. Máscara de Anomalia (com registro)'
        : '3. Máscara de Anomalia (direta)';

      const msgBox = root.querySelector('#sim06_ind_msgStatus');
      const hasMisalignment = (tx !== 0 || ty !== 0 || rot !== 0);
      const misalignmentVisible = hasMisalignment && !modoRegistro;

      if (defectPixelsDetected > 0) {
        if (misalignmentVisible && defectPixelsDetected > 150) {
          msgBox.className = 'sim06_ind_status fail';
          msgBox.innerHTML = '❌ Falsos positivos detectados! O desalinhamento mecânico gerou anomalias fantasmas nas bordas (' + defectPixelsDetected + ' px comprometidos).';
        } else if (misalignmentVisible) {
          msgBox.className = 'sim06_ind_status warn';
          msgBox.innerHTML = '⚠️ Anomalia localizada, mas parte da diferença ainda vem do desalinhamento (' + defectPixelsDetected + ' px afetados).';
        } else {
          msgBox.className = 'sim06_ind_status fail';
          msgBox.innerHTML = '⚠️ Anomalia localizada! Defeito superficial detectado na estrutura interna (' + defectPixelsDetected + ' px afetados).';
        }
      } else {
        if (misalignmentVisible) {
          msgBox.className = 'sim06_ind_status warn';
          msgBox.innerHTML = '✓ Conforme sob risco. O limiar alto omitiu as bordas desalinhadas, mas tornou o sistema cego para defeitos reais.';
        } else {
          msgBox.className = 'sim06_ind_status ok';
          msgBox.innerHTML = '✓ Produto conforme. Sistema corretamente registrado em nível geométrico.';
        }
      }
    }

    ['#sim06_ind_sliderTx', '#sim06_ind_sliderTy', '#sim06_ind_sliderRot', '#sim06_ind_sliderTh'].forEach(sel => {
      root.querySelector(sel).addEventListener('input', update);
    });

    const btnDireto   = root.querySelector('#sim06_ind_btnModoDireto');
    const btnRegistro = root.querySelector('#sim06_ind_btnModoRegistro');

    btnDireto.addEventListener('click', () => {
      modoRegistro = false;
      btnDireto.classList.add('sim06_ind_active');
      btnRegistro.classList.remove('sim06_ind_active');
      update();
    });

    btnRegistro.addEventListener('click', () => {
      modoRegistro = true;
      btnRegistro.classList.add('sim06_ind_active');
      btnDireto.classList.remove('sim06_ind_active');
      update();
    });

    root.querySelector('#sim06_ind_btnReset').addEventListener('click', function() {
      root.querySelector('#sim06_ind_sliderTx').value = 0;
      root.querySelector('#sim06_ind_sliderTy').value = 0;
      root.querySelector('#sim06_ind_sliderRot').value = 0;
      root.querySelector('#sim06_ind_sliderTh').value = 35;
      modoRegistro = false;
      btnDireto.classList.add('sim06_ind_active');
      btnRegistro.classList.remove('sim06_ind_active');
      update();
    });

    update();
  }

  function tryInitSim06Industrial(){
    var root = document.getElementById('sim-06-industrial');
    if (root) initSim06Industrial(root); else setTimeout(tryInitSim06Industrial, 200);
  }
  tryInitSim06Industrial();
})();
</script>
</div>
""")

**Figure 6.19:** Interactive industrial inspection simulator by image subtraction: control the geometric misalignment distortions (registration) and the detection threshold to observe the impact on false positives.


<figure id="fig-06-sim-06-industrial">
  <img src="imagens/fig-06-sim-06-industrial.png" alt=" Interactive industrial inspection simulator by image subtraction: control the geometric misalignment distortions (registration) and the detection threshold to observe the impact on false positives. " style="max-width:80%" />
  <figcaption><strong>Figure 6.19:</strong>  Interactive industrial inspection simulator by image subtraction: control the geometric misalignment distortions (registration) and the detection threshold to observe the impact on false positives. </figcaption>
</figure>

### 6.10.2 Defect Detection by Texture Analysis

In applications where there is no reference image, defect detection can be based on the surface texture characteristics. In this case, the goal is to identify regions whose appearance differs from the predominant pattern of the material.

In this example, the **local variance** is used as a measure of heterogeneity. For each image position, the variance of intensity levels is calculated within a neighborhood of fixed dimensions. Regions with low variance tend to exhibit a more uniform texture, while local alterations, such as scratches, stains, or imperfections, may produce higher values of this measure.

[Figure 6.20](#fig-06-industrial-textura) illustrates this procedure using the image `skimage.data.brick()`. Initially, the local variance map is computed using a sliding window. Subsequently, thresholding is applied to highlight regions whose variance exceeds the specified value, identifying potential areas of interest for inspection.

#### Mathematical Modeling

Consider a grayscale image represented by

$$
f:\Omega\subset\mathbb{Z}^2\rightarrow\mathbb{R},
$$

where $\Omega$ is the image domain and $f(x,y)$ represents the pixel intensity at coordinates $(x,y)$. In 8-bit images, these intensities belong to the interval $[0,255]$. In this example, however, they were normalized to the interval $[0,1]$, without altering the algorithm's operation.

For each position in the image, a square neighborhood $W_{x,y}$ of dimensions $15\times15$ pixels is considered.

The local mean is given by

$$
\mu(x,y)=
\frac{1}{|W_{x,y}|}
\sum_{(u,v)\in W_{x,y}}
f(u,v),
$$

and the local variance is computed as

$$
\sigma^2(x,y) =
\frac{1}{|W_{x,y}|}
\sum_{(u,v)\in W_{x,y}}
f(u,v)^2 -
\mu(x,y)^2.
$$

In the following implementation, these two means are obtained using the `cv2.blur` function,

```python
mean  = cv2.blur(img, (15,15))
mean2 = cv2.blur(img**2, (15,15))
var   = mean2 - mean**2
```

Next, the difference between the variance maps of the reference image and the inspected image is computed,

$$
D(x,y)=
\left|
\sigma_d^2(x,y)-\sigma_r^2(x,y)
\right|,
$$

where $\sigma_r^2(x,y)$ and $\sigma_d^2(x,y)$ are the local variances of the reference image and the image containing the defect, respectively. After normalizing the map $D(x,y)$, a thresholding operation is applied to obtain the mask of possible anomalies.

In [29]:
import numpy as np
import cv2
from skimage import data as skdata
from morph import mm

# Uniform texture image (brick)
texture = skdata.brick().astype(np.float32) / 255.0

# Insertion of synthetic defect: light spot 20×80 px
texture_defect = np.copy(texture)
texture_defect[60:80, 80:160] = 0.95

# Local variance map (window 15×15)
def variancia_local(img, ksize=15):
    img_f = img.astype(np.float32)
    mean  = cv2.blur(img_f, (ksize, ksize))
    mean2 = cv2.blur(img_f ** 2, (ksize, ksize))
    return np.clip(mean2 - mean ** 2, 0, None)

var_ref    = variancia_local(texture)
var_defect = variancia_local(texture_defect)
diff_var   = np.abs(var_defect - var_ref)

# Normalize and threshold
diff_norm = (diff_var / diff_var.max() * 255).astype(np.uint8)
_, mask   = cv2.threshold(diff_norm, 30, 255, cv2.THRESH_BINARY)

mm.show(
    [texture, texture_defect, diff_norm, mask],
    titles=["Original texture", "With defect", "Δ local variance", "Detected anomaly"],
    cols=4,
    figsize=(16, 4)
)

status = "Defeito de textura detectado." if mask.any() else "Superfície conforme."
print(status)


<Figure size 2400x600 with 4 Axes>

**Figure 6.20:** Detection of texture heterogeneity: local variance map and anomaly mask.


Defeito de textura detectado.


> ### 📝 🧠 Why It Works — Texture Analysis
>
> Local variance measures the dispersion of intensities within a neighborhood of the image. In regions where the texture remains uniform, this measure tends to vary little. When a defect modifies the surface pattern, the distribution of intensities also changes, producing differences in local variance.
>
> In this chapter, detection is carried out by comparing the variance maps of the reference image and the defective image. After normalization, thresholding is applied to highlight regions where this difference exceeds a specified value.
>
> The main parameters of the method are the window size used in the variance calculation and the threshold employed in segmentation. Smaller windows are more sensitive to fine details, while larger windows produce smoother maps and may reduce the response to small-sized defects.

## 6.11 Summary

This chapter presented Computer Vision methods applied to document analysis and automated visual inspection. The main techniques studied were:

- **Document preprocessing:** application of background normalization, adaptive equalization (CLAHE), and Otsu thresholding to reduce the effects of non-uniform illumination and improve text segmentation.

- **Optical character recognition (OCR):** conversion of document images into encoded text using Tesseract OCR, highlighting the influence of preprocessing on recognition quality.

- **Machine translation:** application of natural language processing techniques to translate the text obtained by OCR.

- **Geometric document rectification:** use of the Canny edge detector, the Hough Transform, and projective transformations to correct the perspective of scanned documents.

- **Form localization and rectification:** use of morphological operations, contour analysis, and perspective transformation to identify reference markers and automatically extract regions of interest.

- **Reading of two-dimensional and one-dimensional codes:** detection and decoding of *QRCodes* and barcodes for automatic identification of documents and metadata.

- **Optical mark recognition (OMR):** automated reading of forms and answer sheets, illustrated by a case study of the MCTest system.

- **Industrial inspection:** defect detection by comparison with a reference image and by analysis of local variance.

Throughout the chapter, the algorithms were implemented and evaluated with images from the `skimage.data` library and with real documents, allowing the reproduction of the presented experiments.

### Next Steps

The methods presented in this chapter show how Digital Image Processing, Computer Vision, and Natural Language Processing techniques can be integrated into *pipelines* for automated document analysis.

The next chapter will study **feature extraction** and **pattern recognition** techniques, with emphasis on descriptors capable of representing images by numerical attributes for comparison, classification, and automatic recognition. These concepts form the basis for the chapters dedicated to machine learning and deep learning applied to Computer Vision.

## 6.12 🤖 Using Gemini Notebook as a Complementary Tutor

As a study aid for this chapter, the use of **Gemini Notebook** as a complementary tutor is recommended. The tool employs artificial intelligence models to answer questions, prepare summaries, and explain concepts based on the documents provided as reference sources, allowing students to review the content interactively.

> ### ❗ 🎓 Study with the Intelligent Tutor
>
> [🚀 ACCESS Gemini Notebook: CHAPTER 06](https://notebooklm.google.com/notebook/835ec2a8-dbc3-46c4-8c2c-2040af3754a4)
>
> #### 🌐 Language and Programming Language
>
> The project for this chapter in Gemini Notebook was built solely with the text in **Portuguese** and the code examples in **Python**. If you are studying from the English or French edition, or following the C++ track, the tutor's responses may not correspond exactly to the version you are reading.
>
> #### ⚠️ Critical Use of Responses
>
> The responses generated by Gemini Notebook are produced automatically by an artificial intelligence model and may contain omissions or inaccuracies. For this reason, they should be used as supporting material, not as a substitute for studying the chapter.
>
> Whenever questions arise, consult the text of this book, run the examples provided, and, when necessary, supplement your research with books, scientific articles, and other reliable academic sources.

## 6.13 Exercise List

The following exercises consolidate the concepts presented in this chapter through adaptations, experiments, and extensions of the algorithms developed throughout the text.

1. **(10%)** Investigate the influence of the inclination angle in the *deskew* stage. Generate rotated versions of the `skimage.data.page()` image for angles between $-10^\circ$ and $10^\circ$, apply the algorithm presented in the chapter, and compare the estimated angle with the angle used in the rotation. Present the results in a table and discuss the accuracy of the method.

2. **(15%)** Apply background normalization and CLAHE (using at least three combinations of `clipLimit` and `tileGridSize`) to the `skimage.data.page()` image artificially degraded with an illumination gradient and lateral shadow. Segment each version using Otsu's method and compare the results using the number of spurious connected components and the IoU metric relative to a manually constructed reference mask.

3. **(15%)** Investigate the sensitivity of circularity filtering, $C=\frac{4\pi A}{P^2},$ in detecting circular markers. Using the simulator from [Figure 6.9](#fig-06-sim-06-circularidade), generate synthetic disks with increasing geometric noise and evaluate the thresholds $C\in\{0.5,\ 0.6,\ 0.7,\ 0.8\}$. Present a table relating the threshold to the number of false positives and false negatives and discuss the trade-off between sensitivity and specificity.

4. **(15%)** From the four detected markers, implement perspective rectification using `cv2.getPerspectiveTransform` and `cv2.warpPerspective`. Then, artificially perturb the coordinates of the control points with Gaussian noise of standard deviation $\sigma\in\{1,3,5\}$ pixels and evaluate the reprojection error obtained after the inverse homography.

5. **(15%)** Adapt the acquisition and rectification *pipeline* developed in this chapter to process documents containing linear barcodes instead of *QRCodes*. Rasterize the PDF with `pdf2image` at 300 DPI, apply *deskew*, decode the symbol with `pyzbar`, and present the rectified image along with the obtained character sequence.

6. **(15%)** Extend the MCTest bubble reading to identify three situations: **OK**, **BLANK** (no alternative marked), and **DOUBLE MARKING** (two or more alternatives above a fill threshold). Evaluate at least three values of this threshold, present the results in a `pandas.DataFrame`, and discuss its influence on the classification of responses.

7. **(15%)** Build an industrial inspection *pipeline* combining image subtraction and local texture variance analysis on a set of synthetic images containing simulated defects. For each image, generate a reference mask (*ground truth*), compute the IoU metric (*Intersection over Union*) of both approaches for different decision thresholds, and present the results in tables and visualizations produced with `mm.show`.

8. **(Bonus – 10%)** Manually implement the inclination angle estimation without using `cv2.HoughLines` or `cv2.HoughLinesP`. From the edge map obtained by the Canny detector, construct the Hough Transform accumulator, $\rho=x\cos\theta+y\sin\theta,$ for $\theta\in[-45^\circ,45^\circ]$, identify the accumulator maxima, and estimate the inclination by the median of the detected lines. Compare the results with the OpenCV implementation and discuss the influence of spurious lines on the final estimate.

## Chapter References

The theoretical foundation and case studies presented in this chapter are supported by the following references:

- Gonzalez (2018), for the fundamentals of edge detection, thresholding, segmentation, morphological operations, optical character recognition, and geometric transformations applied to document analysis.

- Szeliski (2022), for the Hough Transform, image registration and alignment, projective transformations (homographies), and the fundamentals of automated visual inspection.

- Bradski (2008), for the use of the OpenCV library in the stages of edge detection, Hough Transform, geometric transformations, contour analysis, and QR Code decoding.

- Smith (2007) and Smith (2013), for the architecture, operation, and evolution of the **Tesseract OCR** optical character recognition engine, used in the OCR examples presented in this chapter.

- Bahdanau (2015) and Vaswani (2017), for the fundamentals of neural network-based machine translation, including attention mechanisms and *transformer* architectures.

- Zampirolli (2023), for the description of the MCTest system, used as a case study of a complete pipeline for automated reading and grading of answer sheets.

- Bergmann (2019), Tabernik (2020), and Song (2013), for the public industrial inspection datasets **MVTec AD**, **KolektorSDD**, and **NEU Surface Defect Database**, used as a reference for evaluating and comparing defect detection algorithms.

## Chapter References


BAHDANAU, D.; CHO, K.; BENGIO, Y. **Neural Machine Translation by Jointly Learning to Align and Translate**. 2015.

BERGMANN, P. *et al*. **MVTec AD -- A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection**. 2019.

BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

SMITH, R. **An Overview of the Tesseract OCR Engine**. IEEE Computer Society, 2007.

SMITH, R. **History of the Tesseract OCR Engine: What Worked and What Didn't**. 2013.

SONG, K.; YAN, Y. **A Noise Robust Method Based on Completed Local Binary Patterns for Hot-Rolled Steel Strip Surface Defects**. 2013.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

TABERNIK, D. *et al*. **Segmentation-Based Deep-Learning Approach for Surface-Defect Detection**. 2020.

VASWANI, A. *et al*. **Attention Is All You Need**. 2017.

ZAMPIROLLI, F. A. **MCTest: Como Criar e Corrigir Exames Parametrizados**. Independente, 2023.